In [ ]:
import pandas as pd 

# Load time series data from CSV file
file_path = "/Users/igormol/Desktop/John/0105010006.csv"
data = pd.read_csv(file_path)

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
import seaborn as sns
import matplotlib.pyplot as plt
from tabulate import tabulate

# Define a class for time series modeling
class TimeSeriesModel:
    def __init__(self, data, window_size=1, months_to_predict=12, randomness_factor=0.9):
        # Initialize the model with data and parameters
        self.data = data
        self.window_size = window_size
        self.months_to_predict = months_to_predict
        self.randomness_factor = randomness_factor
        self.predictions = []  # List to store model predictions
        self.future_extrapolation = []  # List to store future extrapolation values

    def preprocess_data(self):
        # Preprocess data by sorting and handling missing values
        self.data = self.data.sort_values(by='TIME_SCALE')
        self.data = self.data.dropna(subset=['COUNT'])

        if self.data['COUNT'].isnull().sum() > 0:
            # Handle NaN values by filling with the mean of the 'COUNT' column
            self.data['COUNT'].fillna(self.data['COUNT'].mean(), inplace=True)

    def train_model(self, X, y):
        # Train a Support Vector Regression (SVR) model
        model = SVR(kernel='rbf', gamma='scale')
        model.fit(X, y)
        return model

    def generate_predictions(self, X, model):
        # Generate predictions using the trained model
        return model.predict(X)

    def sliding_window_predictions(self):
        # Iterate through the DataFrame with a sliding window for historical predictions
        for i in range(len(self.data) - self.window_size + 1):
            window_data = self.data.iloc[i:i + self.window_size]
            X = window_data['TIME_SCALE'].values.reshape(-1, 1)
            y = window_data['COUNT'].values

            if np.isnan(y).any():
                continue  # Skip this iteration if there are NaN values in y

            model = self.train_model(X, y)

            if i + self.window_size < len(self.data):
                next_time_point = self.data.iloc[i + self.window_size]['TIME_SCALE']
                self.future_extrapolation.append(next_time_point)
                prediction = self.generate_predictions([[next_time_point]], model)
                self.predictions.append(prediction[0])

    def extrapolate_future(self):
        # Extrapolate for the specified number of months using a sliding window approach
        for _ in range(self.months_to_predict):
            # Use the last window_data for prediction
            X = self.data['TIME_SCALE'].values.reshape(-1, 1)
            y = self.data['COUNT'].values

            if np.isnan(y).any():
                continue  # Skip this iteration if there are NaN values in y

            model = self.train_model(X, y)

            # Predict the next time point
            next_time_point = self.future_extrapolation[-1] + 1
            self.future_extrapolation.append(next_time_point)
            prediction = self.generate_predictions([[next_time_point]], model)
            # Add some randomness to the prediction
            self.predictions.append(prediction[0] + np.random.normal(scale=self.randomness_factor))

    def calculate_metrics(self):
        # Calculate Mean Absolute Error (MAE), Covariance Matrix, and Standard Deviation
        mae = mean_absolute_error(self.data['COUNT'][:len(self.data)], self.predictions[:len(self.data)])
        covariance_matrix = np.cov(self.data['COUNT'][:len(self.data)], self.predictions[:len(self.data)])
        std_deviation = np.std(self.data['COUNT'][:len(self.data)] - self.predictions[:len(self.data)])
        return mae, covariance_matrix, std_deviation

    def plot_results(self):
        # Use Seaborn for styling
        sns.set(style="whitegrid")
        plt.figure(figsize=(12, 6))

        # Combine model prediction and future extrapolation into a single curve
        combined_time = np.concatenate((self.data['TIME_SCALE'], self.future_extrapolation))
        combined_predictions = np.concatenate((self.data['PREDICTION'], self.predictions))

        # Plot the original data and the combined curve
        sns.lineplot(x=self.data['TIME_SCALE'], y=self.data['COUNT'], label='Original Data', color='blue')
        sns.lineplot(x=combined_time, y=combined_predictions,
                     label='Model Prediction and Future Extrapolation', linestyle='dashed', color='green', marker='o')

        plt.xlabel('Time Scale')
        plt.ylabel('Count')
        plt.title(f'Time Series Modeling with Sliding Window and {self.months_to_predict} Months Future Extrapolation (SVM)')
        plt.legend()
        plt.show()

    def display_results_table(self, mae, covariance_matrix, std_deviation):
        # Display results in a table
        results_table = pd.DataFrame({
            'Metric': ['Mean Absolute Error (MAE)', 'Covariance Matrix', 'Standard Deviation'],
            'Value': [mae, covariance_matrix, std_deviation]
        })

        # Print the table using tabulate
        print(tabulate(results_table, headers='keys', tablefmt='fancy_grid'))


def main():
    # Assuming your DataFrame is named 'data'
    # If not, replace 'data' with your actual DataFrame name
    time_series_model = TimeSeriesModel(data)
    time_series_model.preprocess_data()
    time_series_model.sliding_window_predictions()
    time_series_model.extrapolate_future()
    mae, covariance_matrix, std_deviation = time_series_model.calculate_metrics()
    time_series_model.plot_results()
    time_series_model.display_results_table(mae, covariance_matrix, std_deviation)


if __name__ == "__main__":
    main()


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
import seaborn as sns
import matplotlib.pyplot as plt
from tabulate import tabulate

# Define a class for time series modeling
class TimeSeriesModel:
    def __init__(self, data, window_size=1, months_to_predict=12, randomness_factor=0.9):
        # Initialize the model with data and parameters
        self.data = data
        self.window_size = window_size
        self.months_to_predict = months_to_predict
        self.randomness_factor = randomness_factor
        self.predictions = []  # List to store model predictions
        self.future_extrapolation = []  # List to store future extrapolation values

    def preprocess_data(self):
        # Preprocess data by sorting and handling missing values
        self.data = self.data.sort_values(by='TIME_SCALE')
        self.data = self.data.dropna(subset=['COUNT'])

        if self.data['COUNT'].isnull().sum() > 0:
            # Handle NaN values by filling with the mean of the 'COUNT' column
            self.data['COUNT'].fillna(self.data['COUNT'].mean(), inplace=True)

    def train_model(self, X, y):
        # Train a Support Vector Regression (SVR) model
        model = SVR(kernel='rbf', gamma='scale')
        model.fit(X, y)
        return model

    def generate_predictions(self, X, model):
        # Generate predictions using the trained model
        return model.predict(X)

    def sliding_window_predictions(self):
        # Iterate through the DataFrame with a sliding window for historical predictions
        for i in range(len(self.data) - self.window_size + 1):
            window_data = self.data.iloc[i:i + self.window_size]
            X = window_data['TIME_SCALE'].values.reshape(-1, 1)
            y = window_data['COUNT'].values

            if np.isnan(y).any():
                continue  # Skip this iteration if there are NaN values in y

            model = self.train_model(X, y)

            if i + self.window_size < len(self.data):
                next_time_point = self.data.iloc[i + self.window_size]['TIME_SCALE']
                self.future_extrapolation.append(next_time_point)
                prediction = self.generate_predictions([[next_time_point]], model)
                self.predictions.append(prediction[0])

    def extrapolate_future(self):
        # Extrapolate for the specified number of months using a sliding window approach
        for _ in range(self.months_to_predict):
            # Use the last window_data for prediction
            X = self.data['TIME_SCALE'].values.reshape(-1, 1)
            y = self.data['COUNT'].values

            if np.isnan(y).any():
                continue  # Skip this iteration if there are NaN values in y

            model = self.train_model(X, y)

            # Predict the next time point
            next_time_point = self.future_extrapolation[-1] + 1
            self.future_extrapolation.append(next_time_point)
            prediction = self.generate_predictions([[next_time_point]], model)
            # Add some randomness to the prediction
            self.predictions.append(prediction[0] + np.random.normal(scale=self.randomness_factor))

    def calculate_metrics(self):
        # Calculate Mean Absolute Error (MAE), Covariance Matrix, and Standard Deviation
        mae = mean_absolute_error(self.data['COUNT'][:len(self.data)], self.predictions[:len(self.data)])
        covariance_matrix = np.cov(self.data['COUNT'][:len(self.data)], self.predictions[:len(self.data)])
        std_deviation = np.std(self.data['COUNT'][:len(self.data)] - self.predictions[:len(self.data)])
        return mae, covariance_matrix, std_deviation

    def plot_results(self):
        # Use Seaborn for styling
        sns.set(style="whitegrid")
        plt.figure(figsize=(12, 6))

        # Combine model prediction and future extrapolation into a single curve
        combined_time = np.concatenate((self.data['TIME_SCALE'], self.future_extrapolation))
        combined_predictions = np.concatenate((self.data['PREDICTION'], self.predictions))

        # Plot the original data and the combined curve
        sns.lineplot(x=self.data['TIME_SCALE'], y=self.data['COUNT'], label='Original Data', color='blue')
        sns.lineplot(x=combined_time, y=combined_predictions,
                     label='Model Prediction and Future Extrapolation', linestyle='dashed', color='green', marker='o')

        plt.xlabel('Time Scale')
        plt.ylabel('Count')
        plt.title(f'Time Series Modeling with Sliding Window and {self.months_to_predict} Months Future Extrapolation (SVM)')
        plt.legend()
        plt.show()

    def display_results_table(self, mae, covariance_matrix, std_deviation):
        # Display results in a table
        results_table = pd.DataFrame({
            'Metric': ['Mean Absolute Error (MAE)', 'Covariance Matrix', 'Standard Deviation'],
            'Value': [mae, covariance_matrix, std_deviation]
        })

        # Print the table using tabulate
        print(tabulate(results_table, headers='keys', tablefmt='fancy_grid'))


def main():
    # Assuming your DataFrame is named 'data'
    # If not, replace 'data' with your actual DataFrame name
    time_series_model = TimeSeriesModel(data)
    time_series_model.preprocess_data()
    time_series_model.sliding_window_predictions()
    time_series_model.extrapolate_future()
    mae, covariance_matrix, std_deviation = time_series_model.calculate_metrics()
    time_series_model.plot_results()
    time_series_model.display_results_table(mae, covariance_matrix, std_deviation)


if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
import seaborn as sns
import matplotlib.pyplot as plt
from tabulate import tabulate

class TimeSeriesModel:
    def __init__(self, data, window_size=1, months_to_predict=12, randomness_factor=0.9):
        self.data = data
        self.window_size = window_size
        self.months_to_predict = months_to_predict
        self.randomness_factor = randomness_factor
        self.predictions = []
        self.future_extrapolation = []

    def preprocess_data(self):
        self.data = self.data.sort_values(by='TIME_SCALE')
        self.data = self.data.dropna(subset=['COUNT'])

        if self.data['COUNT'].isnull().sum() > 0:
            self.data['COUNT'].fillna(self.data['COUNT'].mean(), inplace=True)

    def train_model(self, X, y):
        model = SVR(kernel='rbf', gamma='scale')
        model.fit(X, y)
        return model

    def generate_predictions(self, X, model):
        return model.predict(X)

    def sliding_window_predictions(self):
        for i in range(len(self.data) - self.window_size + 1):
            window_data = self.data.iloc[i:i + self.window_size]
            X = window_data['TIME_SCALE'].values.reshape(-1, 1)
            y = window_data['COUNT'].values

            if np.isnan(y).any():
                continue

            model = self.train_model(X, y)

            if i + self.window_size < len(self.data):
                next_time_point = self.data.iloc[i + self.window_size]['TIME_SCALE']
                self.future_extrapolation.append(next_time_point)
                prediction = self.generate_predictions([[next_time_point]], model)
                self.predictions.append(prediction[0])

                # Add 'PREDICTION' column to the DataFrame
                self.data.at[i + self.window_size, 'PREDICTION'] = prediction[0]

    def extrapolate_future(self):
        for _ in range(self.months_to_predict):
            X = self.data['TIME_SCALE'].values.reshape(-1, 1)
            y = self.data['COUNT'].values

            if np.isnan(y).any():
                continue

            model = self.train_model(X, y)

            next_time_point = self.future_extrapolation[-1] + 1
            self.future_extrapolation.append(next_time_point)
            prediction = self.generate_predictions([[next_time_point]], model)
            self.predictions.append(prediction[0] + np.random.normal(scale=self.randomness_factor))

    def calculate_metrics(self):
        mae = mean_absolute_error(self.data['COUNT'][:len(self.data)], self.predictions[:len(self.data)])
        covariance_matrix = np.cov(self.data['COUNT'][:len(self.data)], self.predictions[:len(self.data)])
        std_deviation = np.std(self.data['COUNT'][:len(self.data)] - self.predictions[:len(self.data)])
        return mae, covariance_matrix, std_deviation

    def plot_results(self):
        sns.set(style="whitegrid")
        plt.figure(figsize=(12, 6))

        combined_time = np.concatenate((self.data['TIME_SCALE'], self.future_extrapolation))
        combined_predictions = np.concatenate((self.data['PREDICTION'], self.predictions))

        sns.lineplot(x=self.data['TIME_SCALE'], y=self.data['COUNT'], label='Original Data', color='blue')
        sns.lineplot(x=combined_time, y=combined_predictions,
                     label='Model Prediction and Future Extrapolation', linestyle='dashed', color='green', marker='o')

        plt.xlabel('Time Scale')
        plt.ylabel('Count')
        plt.title(f'Time Series Modeling with Sliding Window and {self.months_to_predict} Months Future Extrapolation (SVM)')
        plt.legend()
        plt.show()

    def display_results_table(self, mae, covariance_matrix, std_deviation):
        results_table = pd.DataFrame({
            'Metric': ['Mean Absolute Error (MAE)', 'Covariance Matrix', 'Standard Deviation'],
            'Value': [mae, covariance_matrix, std_deviation]
        })

        print(tabulate(results_table, headers='keys', tablefmt='fancy_grid'))


def main():
    # Assuming your DataFrame is named 'data'
    # If not, replace 'data' with your actual DataFrame name
    time_series_model = TimeSeriesModel(data)
    time_series_model.preprocess_data()
    time_series_model.sliding_window_predictions()
    time_series_model.extrapolate_future()
    mae, covariance_matrix, std_deviation = time_series_model.calculate_metrics()
    time_series_model.plot_results()
    time_series_model.display_results_table(mae, covariance_matrix, std_deviation)


if __name__ == "__main__":
    main()


In [ ]:
# Hybrid Deep Learning Approach (RNN and LSTM layers) for Inventory Forecasting
#
# Igor Mol <igor.mol@makes.ai>
#
# The provided code is a comprehensive Python script for time series forecasting
# using a recurrent neural network (RNN) with Long Short-Term Memory (LSTM) lay-
# ers. It follows a structured approach, encapsulating functionality within fun-
# ctions and a class for better organization. The first set of functions revol-
# ves around data handling: load_data loads a CSV file into a Pandas DataFrame,
# while preprocess_data extracts the relevant features and target variables, re-
# serves 80% of the data for training, and reshapes the input data for the RNN.
#     The subsequent functions are focused on constructing and training the deep
# learning model. build_model creates a Sequential model with an architecture 
# comprising multiple LSTM layers and a dense output layer. The train_model fun-
# ction utilizes the training data to fit the model, specifying the number of 
# epochs and batch size. The training history is recorded for later analysis. The
# make_predictions function employs the trained model to predict the entire time
# interval of the original data and extrapolate future values. It returns the 
# model predictions for both the historical data and the future time range.
#     Moving on to evaluation, the calculate_mae function computes the Mean 
# Absolute Error (MAE) between the true target values and the predictions, pro-
# viding a quantitative measure of model performance. Finally, the plot_results 
# function generates a visually informative plot, displaying the original time 
# series, model predictions, and extrapolated future predictions. It also inclu-
# des a subplot illustrating the Mean Squared Error (MSE) over training epochs,
# offering insights into the convergence and performance of the neural network.
# This structured and modular approach enhances code readability and facilitates
# reuse in different scenarios.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

def load_data(file_path):
    # Load CSV file into the data DataFrame
    data = pd.read_csv(file_path)
    return data

def preprocess_data(data):
    # Extracting the features and target variable
    X = data['TIME_SCALE'].values
    y = data['COUNT'].values

    # Use 80% of the data for training
    split_index = int(0.8 * len(X))
    X_train = X[:split_index]
    y_train = y[:split_index]

    # Reshape the input data for RNN
    X_train = X_train.reshape((X_train.shape[0], 1, 1))

    return X_train, y_train, X, y

def build_model():
    # Building a deeper and more complex RNN model
    model = Sequential()
    model.add(LSTM(units=100, activation='relu', input_shape=(1, 1), return_sequences=True))
    model.add(LSTM(units=200, activation='relu', return_sequences=True))
    model.add(LSTM(units=300, activation='relu', return_sequences=True))
    model.add(Dropout(0.2))
    model.add(LSTM(units=200, activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(units=1))
    model.compile(optimizer='adam', loss='mean_squared_error')

    return model

def train_model(model, X_train, y_train, epochs=200, batch_size=1):
    # Training the model on 80% of the time interval
    history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, verbose=2)
    return history

def make_predictions(model, X, smoothing_factor=0.1, randomness_factor=0.9):
    # Use the entire time interval for predictions
    X = X.reshape((X.shape[0], 1, 1))
    predictions = model.predict(X)

    # Initialize the smoothed prediction with the first prediction
    smoothed_predictions = [predictions[0]]

    # Apply exponential smoothing to the model predictions with added randomness
    for i in range(1, len(predictions)):
        noise = np.random.normal(scale=randomness_factor)
        smoothed_value = smoothing_factor * (predictions[i] + noise) + (1 - smoothing_factor) * smoothed_predictions[-1]
        smoothed_predictions.append(smoothed_value)

    # Extrapolate to predict the future 12 months ahead
    future_time_range = np.arange(max(X), max(X) + 13, 1)  # Extend by 12 months
    future_X = future_time_range.reshape((len(future_time_range), 1, 1))
    future_predictions = model.predict(future_X)

    # Initialize the smoothed future prediction with the last smoothed model prediction
    smoothed_future_predictions = [smoothed_predictions[-1]]

    # Apply exponential smoothing to the future predictions with added randomness
    for i in range(1, len(future_predictions)):
        noise = np.random.normal(scale=randomness_factor)
        smoothed_value = smoothing_factor * (future_predictions[i] + noise) + (1 - smoothing_factor) * smoothed_future_predictions[-1]
        smoothed_future_predictions.append(smoothed_value)

    return np.array(smoothed_predictions), future_time_range, np.array(smoothed_future_predictions)


def calculate_mae(y_true, predictions):
    # Calculate MAE
    mae = mean_absolute_error(y_true, predictions.flatten())
    print(f'Mean Absolute Error (MAE): {mae}')

def plot_results(data, predictions, future_time_range, future_predictions, history):
    # Plotting the results with improved quality
    plt.figure(figsize=(12, 8))

    # Plot Original Data and Predictions
    plt.subplot(2, 1, 1)
    plt.plot(data['TIME_SCALE'], data['COUNT'], label='Original Data', marker='o', linestyle='-')
    plt.plot(data['TIME_SCALE'], predictions, label='Model Prediction', linestyle='dashed', color='orange')
    plt.plot(future_time_range, future_predictions, label='Future Prediction (12 months)', linestyle='dashed', color='green')
    plt.title('RNN Model vs Original Data (Extrapolation)')
    plt.xlabel('Time')
    plt.ylabel('COUNT')
    plt.legend()
    plt.grid(True)

    # Plot MSE over training epochs
    plt.subplot(2, 1, 2)
    plt.plot(history.history['loss'], label='Mean Squared Error (MSE)')
    plt.title('Mean Squared Error (MSE) Over Training Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('MSE')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

# File path for CSV
#file_path = "/Users/igormol/Desktop/John/John.csv"

# Load data
#data = load_data(file_path)

# Preprocess data
X_train, y_train, X, y = preprocess_data(data)

# Build model
model = build_model()

# Train model
history = train_model(model, X_train, y_train)

# Make predictions with exponential smoothing
predictions, future_time_range, future_predictions = make_predictions(model, X)
smoothed_predictions, future_time_range, smoothed_future_predictions = make_predictions(model, X, smoothing_factor=0.1)

# Calculate MAE for smoothed predictions
calculate_mae(y, smoothed_predictions)

# Plot results with smoothed predictions
plot_results(data, smoothed_predictions, future_time_range, smoothed_future_predictions, history)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

def load_data(file_path):
    # Load CSV file into the data DataFrame
    data = pd.read_csv(file_path)
    return data

def preprocess_data(data):
    # Extracting the features and target variable
    X = data['TIME_SCALE'].values
    y = data['COUNT'].values

    # Use 80% of the data for training
    split_index = int(0.8 * len(X))
    X_train = X[:split_index]
    y_train = y[:split_index]

    # Reshape the input data for RNN
    X_train = X_train.reshape((X_train.shape[0], 1, 1))

    return X_train, y_train, X, y

def build_model():
    # Building a deeper and more complex RNN model
    model = Sequential()
    model.add(LSTM(units=100, activation='relu', input_shape=(1, 1), return_sequences=True))
    model.add(LSTM(units=200, activation='relu', return_sequences=True))
    model.add(LSTM(units=300, activation='relu', return_sequences=True))
    model.add(Dropout(0.2))
    model.add(LSTM(units=200, activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(units=1))
    model.compile(optimizer='adam', loss='mean_squared_error')

    return model

def train_model(model, X_train, y_train, epochs=200, batch_size=1):
    # Training the model on 80% of the time interval
    history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, verbose=2)
    return history

def make_predictions(model, X, smoothing_factor=0.1, randomness_factor=0.1):
    # Use the entire time interval for predictions
    X = X.reshape((X.shape[0], 1, 1))
    predictions = model.predict(X)

    # Initialize the smoothed prediction with the first prediction
    smoothed_predictions = [predictions[0]]

    # Apply exponential smoothing to the model predictions with added randomness
    for i in range(1, len(predictions)):
        noise = np.random.normal(scale=randomness_factor)
        smoothed_value = smoothing_factor * (predictions[i] + noise) + (1 - smoothing_factor) * smoothed_predictions[-1]
        smoothed_predictions.append(smoothed_value)

    # Extrapolate to predict the future 12 months ahead
    future_time_range = np.arange(max(X), max(X) + 13, 1)  # Extend by 12 months
    future_X = future_time_range.reshape((len(future_time_range), 1, 1))
    future_predictions = model.predict(future_X)

    # Initialize the smoothed future prediction with the last smoothed model prediction
    smoothed_future_predictions = [smoothed_predictions[-1]]

    # Apply exponential smoothing to the future predictions with added randomness
    for i in range(1, len(future_predictions)):
        noise = np.random.normal(scale=randomness_factor)
        smoothed_value = smoothing_factor * (future_predictions[i] + noise) + (1 - smoothing_factor) * smoothed_future_predictions[-1]
        smoothed_future_predictions.append(smoothed_value)

    return np.array(smoothed_predictions), future_time_range, np.array(smoothed_future_predictions)

def calculate_mae(y_true, predictions):
    # Calculate MAE
    mae = mean_absolute_error(y_true, predictions.flatten())
    print(f'Mean Absolute Error (MAE): {mae}')

def plot_results(data, predictions, future_time_range, future_predictions, history):
    # Plotting the results with improved quality
    plt.figure(figsize=(12, 8))

    # Plot Original Data and Predictions
    plt.subplot(2, 1, 1)
    plt.plot(data['TIME_SCALE'], data['COUNT'], label='Original Data', marker='o', linestyle='-')
    plt.plot(data['TIME_SCALE'], predictions, label='Model Prediction', linestyle='dashed', color='orange')
    plt.plot(future_time_range, future_predictions, label='Future Prediction (12 months)', linestyle='dashed', color='green')
    plt.title('RNN Model vs Original Data (Extrapolation)')
    plt.xlabel('Time')
    plt.ylabel('COUNT')
    plt.legend()
    plt.grid(True)

    # Plot MSE over training epochs
    plt.subplot(2, 1, 2)
    plt.plot(history.history['loss'], label='Mean Squared Error (MSE)')
    plt.title('Mean Squared Error (MSE) Over Training Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('MSE')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

# File path for CSV
# file_path = "/Users/igormol/Desktop/John/John.csv"

# Load data
# data = load_data(file_path)

# Preprocess data
X_train, y_train, X, y = preprocess_data(data)

# Build model
model = build_model()

# Train model
history = train_model(model, X_train, y_train)

# Make predictions with exponential smoothing and randomness
predictions, future_time_range, future_predictions = make_predictions(model, X, randomness_factor=0.1)

# Calculate MAE for smoothed predictions
calculate_mae(y, predictions)

# Plot results with smoothed predictions
plot_results(data, predictions, future_time_range, future_predictions, history)


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# Function to load data from a CSV file into a Pandas DataFrame
def load_data(file_path):
    data = pd.read_csv(file_path)
    return data

# Function to preprocess data, extract features and target variable, and reshape input for RNN
def preprocess_data(data):
    X = data['TIME_SCALE'].values
    y = data['COUNT'].values

    # Use 80% of the data for training
    split_index = int(0.8 * len(X))
    X_train = X[:split_index]
    y_train = y[:split_index]

    # Reshape the input data for RNN
    X_train = X_train.reshape((X_train.shape[0], 1, 1))

    return X_train, y_train, X, y

# Function to build a deep and complex RNN model using Keras Sequential API
def build_model():
    model = Sequential()
    model.add(LSTM(units=100, activation='relu', input_shape=(1, 1), return_sequences=True))
    model.add(LSTM(units=200, activation='relu', return_sequences=True))
    model.add(LSTM(units=300, activation='relu', return_sequences=True))
    model.add(Dropout(0.2))
    model.add(LSTM(units=200, activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(units=1))
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

# Function to train the RNN model on 80% of the time interval
def train_model(model, X_train, y_train, epochs=200, batch_size=1):
    history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, verbose=2)
    return history

# Function to make predictions using the trained model with exponential smoothing and randomness
def make_predictions(model, X, smoothing_factor=0.1, randomness_factor=0.1):
    X = X.reshape((X.shape[0], 1, 1))
    predictions = model.predict(X)

    # Initialize the smoothed prediction with the first prediction
    smoothed_predictions = [predictions[0]]

    # Apply exponential smoothing to the model predictions with added randomness
    for i in range(1, len(predictions)):
        noise = np.random.normal(scale=randomness_factor)
        smoothed_value = smoothing_factor * (predictions[i] + noise) + (1 - smoothing_factor) * smoothed_predictions[-1]
        smoothed_predictions.append(smoothed_value)

    # Extrapolate to predict the future 12 months ahead
    future_time_range = np.arange(max(X), max(X) + 13, 1)  # Extend by 12 months
    future_X = future_time_range.reshape((len(future_time_range), 1, 1))
    future_predictions = model.predict(future_X)

    # Initialize the smoothed future prediction with the last smoothed model prediction
    smoothed_future_predictions = [smoothed_predictions[-1]]

    # Apply exponential smoothing to the future predictions with added randomness
    for i in range(1, len(future_predictions)):
        noise = np.random.normal(scale=randomness_factor)
        smoothed_value = smoothing_factor * (future_predictions[i] + noise) + (1 - smoothing_factor) * smoothed_future_predictions[-1]
        smoothed_future_predictions.append(smoothed_value)

    return np.array(smoothed_predictions), future_time_range, np.array(smoothed_future_predictions)

# Function to calculate Mean Absolute Error (MAE) between true target values and predictions
def calculate_mae(y_true, predictions):
    mae = mean_absolute_error(y_true, predictions.flatten())
    print(f'Mean Absolute Error (MAE): {mae}')

# Function to plot the results, including original data, model predictions, and MSE over training epochs
def plot_results(data, predictions, future_time_range, future_predictions, history):
    plt.figure(figsize=(12, 8))

    # Plot Original Data and Predictions
    plt.subplot(2, 1, 1)
    plt.plot(data['TIME_SCALE'], data['COUNT'], label='Original Data', marker='o', linestyle='-')
    plt.plot(data['TIME_SCALE'], predictions, label='Model Prediction', linestyle='dashed', color='orange')
    plt.plot(future_time_range, future_predictions, label='Future Prediction (12 months)', linestyle='dashed', color='green')
    plt.title('RNN Model vs Original Data (Extrapolation)')
    plt.xlabel('Time')
    plt.ylabel('COUNT')
    plt.legend()
    plt.grid(True)

    # Plot MSE over training epochs
    plt.subplot(2, 1, 2)
    plt.plot(history.history['loss'], label='Mean Squared Error (MSE)')
    plt.title('Mean Squared Error (MSE) Over Training Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('MSE')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

# Uncomment the following lines to run the code with your specific file path and data
# file_path = "/Users/igormol/Desktop/John/John.csv"
# data = load_data(file_path)
# X_train, y_train, X, y = preprocess_data(data)
# model = build_model()
# history = train_model(model, X_train, y_train)
# predictions, future_time_range, future_predictions = make_predictions(model, X, randomness_factor=0.1)
# calculate_mae(y, predictions)
# plot_results(data, predictions, future_time_range, future_predictions, history)


In [ ]:
# Hybrid Deep Learning Approach (RNN and LSTM layers) for Inventory Forecasting
#
# Igor Mol <igor.mol@makes.ai>
#
# The provided code is a comprehensive Python script for time series forecasting
# using a recurrent neural network (RNN) with Long Short-Term Memory (LSTM) lay-
# ers. It follows a structured approach, encapsulating functionality within fun-
# ctions and a class for better organization. The first set of functions revol-
# ves around data handling: load_data loads a CSV file into a Pandas DataFrame,
# while preprocess_data extracts the relevant features and target variables, re-
# serves 80% of the data for training, and reshapes the input data for the RNN.
#     The subsequent functions are focused on constructing and training the deep
# learning model. build_model creates a Sequential model with an architecture 
# comprising multiple LSTM layers and a dense output layer. The train_model fun-
# ction utilizes the training data to fit the model, specifying the number of 
# epochs and batch size. The training history is recorded for later analysis. The
# make_predictions function employs the trained model to predict the entire time
# interval of the original data and extrapolate future values. It returns the 
# model predictions for both the historical data and the future time range.
#     Moving on to evaluation, the calculate_mae function computes the Mean 
# Absolute Error (MAE) between the true target values and the predictions, pro-
# viding a quantitative measure of model performance. Finally, the plot_results 
# function generates a visually informative plot, displaying the original time 
# series, model predictions, and extrapolated future predictions. It also inclu-
# des a subplot illustrating the Mean Squared Error (MSE) over training epochs,
# offering insights into the convergence and performance of the neural network.
# This structured and modular approach enhances code readability and facilitates
# reuse in different scenarios.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

def load_data(file_path):
    # Load CSV file into the data DataFrame
    data = pd.read_csv(file_path)
    return data

def preprocess_data(data):
    # Extracting the features and target variable
    X = data['TIME_SCALE'].values
    y = data['COUNT'].values

    # Use 80% of the data for training
    split_index = int(0.8 * len(X))
    X_train = X[:split_index]
    y_train = y[:split_index]

    # Reshape the input data for RNN
    X_train = X_train.reshape((X_train.shape[0], 1, 1))

    return X_train, y_train, X, y

def build_model():
    # Building a deeper and more complex RNN model
    model = Sequential()
    model.add(LSTM(units=100, activation='relu', input_shape=(1, 1), return_sequences=True))
    model.add(LSTM(units=200, activation='relu', return_sequences=True))
    model.add(LSTM(units=300, activation='relu', return_sequences=True))
    model.add(Dropout(0.2))
    model.add(LSTM(units=200, activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(units=1))
    model.compile(optimizer='adam', loss='mean_squared_error')

    return model

def train_model(model, X_train, y_train, epochs=200, batch_size=1):
    # Training the model on 80% of the time interval
    history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, verbose=2)
    return history

def make_predictions(model, X, smoothing_factor=0.1, randomness_factor=0.1):
    # Use the entire time interval for predictions
    X = X.reshape((X.shape[0], 1, 1))
    predictions = model.predict(X)

    # Initialize the smoothed prediction with the first prediction
    smoothed_predictions = [predictions[0]]

    # Apply exponential smoothing to the model predictions with added randomness
    for i in range(1, len(predictions)):
        noise = np.random.normal(scale=randomness_factor)
        smoothed_value = smoothing_factor * (predictions[i] + noise) + (1 - smoothing_factor) * smoothed_predictions[-1]
        smoothed_predictions.append(smoothed_value)

    # Extrapolate to predict the future 12 months ahead
    future_time_range = np.arange(max(X), max(X) + 13, 1)  # Extend by 12 months
    future_X = future_time_range.reshape((len(future_time_range), 1, 1))
    future_predictions = model.predict(future_X)

    # Initialize the smoothed future prediction with the last smoothed model prediction
    smoothed_future_predictions = [smoothed_predictions[-1]]

    # Apply exponential smoothing to the future predictions with added randomness
    for i in range(1, len(future_predictions)):
        noise = np.random.normal(scale=randomness_factor)
        smoothed_value = smoothing_factor * (future_predictions[i] + noise) + (1 - smoothing_factor) * smoothed_future_predictions[-1]
        smoothed_future_predictions.append(smoothed_value)

    return np.array(smoothed_predictions), future_time_range, np.array(smoothed_future_predictions)

def calculate_mae(y_true, predictions):
    # Calculate MAE
    mae = mean_absolute_error(y_true, predictions.flatten())
    print(f'Mean Absolute Error (MAE): {mae}')

def plot_results(data, predictions, future_time_range, future_predictions, history):
    # Plotting the results with improved quality
    plt.figure(figsize=(12, 8))

    # Plot Original Data and Predictions
    plt.subplot(2, 1, 1)
    plt.plot(data['TIME_SCALE'], data['COUNT'], label='Original Data', marker='o', linestyle='-')
    plt.plot(data['TIME_SCALE'], predictions, label='Model Prediction', linestyle='dashed', color='orange')
    plt.plot(future_time_range, future_predictions, label='Future Prediction (12 months)', linestyle='dashed', color='green')
    plt.title('RNN Model vs Original Data (Extrapolation)')
    plt.xlabel('Time')
    plt.ylabel('COUNT')
    plt.legend()
    plt.grid(True)

    # Plot MSE over training epochs
    plt.subplot(2, 1, 2)
    plt.plot(history.history['loss'], label='Mean Squared Error (MSE)')
    plt.title('Mean Squared Error (MSE) Over Training Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('MSE')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

# File path for CSV
# file_path = "/Users/igormol/Desktop/John/John.csv"

# Load data
# data = load_data(file_path)

# Preprocess data
X_train, y_train, X, y = preprocess_data(data)

# Build model
model = build_model()

# Train model
history = train_model(model, X_train, y_train)

# Make predictions with exponential smoothing and randomness
predictions, future_time_range, future_predictions = make_predictions(model, X, randomness_factor=0.1)

# Calculate MAE for smoothed predictions
calculate_mae(y, predictions)

# Plot results with smoothed predictions
plot_results(data, predictions, future_time_range, future_predictions, history)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
order = (p, d, q)  # You need to specify appropriate values for p, d, and q
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 5
forecast = result.get_forecast(steps=future_steps)

# Plot original data
plt.plot(data['TIME_SCALE'], data['COUNT'], label='Original Data')

# Plot model
plt.plot(data['TIME_SCALE'], result.fittedvalues, color='red', label='Fitted Model')

# Plot future extrapolation
plt.plot(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps),
         forecast.predicted_mean, color='green', label='Future Extrapolation')

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data')
plt.legend()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 5
forecast = result.get_forecast(steps=future_steps)

# Plot original data
plt.plot(data['TIME_SCALE'], data['COUNT'], label='Original Data')

# Plot model
plt.plot(data['TIME_SCALE'], result.fittedvalues, color='red', label='Fitted Model')

# Plot future extrapolation
plt.plot(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps),
         forecast.predicted_mean, color='green', label='Future Extrapolation')

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data')
plt.legend()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 5
forecast = result.get_forecast(steps=future_steps)

# Concatenate fitted values and future extrapolation values
fitted_and_forecasted = pd.concat([result.fittedvalues, forecast.predicted_mean])

# Plot original data, fitted model, and future extrapolation
plt.plot(data['TIME_SCALE'], data['COUNT'], label='Original Data')
plt.plot(data['TIME_SCALE'], fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation')

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data')
plt.legend()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 5
forecast = result.get_forecast(steps=future_steps)

# Concatenate fitted values and future extrapolation values with a new index
fitted_and_forecasted = pd.concat([result.fittedvalues, forecast.predicted_mean])
new_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))
fitted_and_forecasted = fitted_and_forecasted.reindex(new_index)

# Plot original data, fitted model, and future extrapolation
plt.plot(data['TIME_SCALE'], data['COUNT'], label='Original Data')
plt.plot(new_index, fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation')

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data')
plt.legend()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 5
forecast = result.get_forecast(steps=future_steps)

# Concatenate fitted values and future extrapolation values with a new index
fitted_and_forecasted = pd.concat([result.fittedvalues, forecast.predicted_mean])
new_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))
fitted_and_forecasted = fitted_and_forecasted.reindex(new_index)

# Plot original data, fitted model, and future extrapolation
plt.plot(data['TIME_SCALE'], data['COUNT'], label='Original Data')
plt.plot(new_index, fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation')

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data')
plt.legend()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 12
forecast = result.get_forecast(steps=future_steps)

# Create a new index for the entire time range
full_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))

# Concatenate fitted values and future extrapolation values with the new index
fitted_and_forecasted = pd.concat([result.fittedvalues, forecast.predicted_mean], axis=0)
fitted_and_forecasted.index = full_index

# Plot original data, fitted model, and future extrapolation
plt.plot(data['TIME_SCALE'], data['COUNT'], label='Original Data')
plt.plot(fitted_and_forecasted.index, fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation')

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data')
plt.legend()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 5
forecast = result.get_forecast(steps=future_steps)

# Get mean and confidence intervals for the future extrapolation
mean_values = forecast.predicted_mean
confidence_intervals = forecast.conf_int()

# Add randomness by sampling from the confidence interval
randomness = np.random.uniform(low=confidence_intervals.iloc[:, 0], high=confidence_intervals.iloc[:, 1])

# Plot original data, fitted model, and future extrapolation with randomness
plt.plot(data['TIME_SCALE'], data['COUNT'], label='Original Data')
plt.plot(result.fittedvalues.index, result.fittedvalues, color='red', label='Fitted Model')
plt.plot(mean_values.index, randomness, color='green', label='Future Extrapolation with Randomness')

plt.fill_between(confidence_intervals.index, confidence_intervals.iloc[:, 0], confidence_intervals.iloc[:, 1], color='green', alpha=0.2)

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data with Randomness in Future Extrapolation')
plt.legend()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 5
forecast = result.get_forecast(steps=future_steps)

# Get mean and confidence intervals for the future extrapolation
mean_values = forecast.predicted_mean
confidence_intervals = forecast.conf_int()

# Add randomness by sampling from the confidence interval
randomness = np.random.uniform(low=confidence_intervals.iloc[:, 0], high=confidence_intervals.iloc[:, 1])

# Plot original data, fitted model, and future extrapolation with randomness
plt.plot(data['TIME_SCALE'], data['COUNT'], label='Original Data')
plt.plot(result.fittedvalues.index, result.fittedvalues, color='red', label='Fitted Model')

# Plot future extrapolation with randomness
future_index = pd.RangeIndex(start=data['TIME_SCALE'].max() + 1, stop=data['TIME_SCALE'].max() + 1 + future_steps)
plt.plot(future_index, randomness, color='green', label='Future Extrapolation with Randomness')

# Fill the area between the confidence intervals
plt.fill_between(confidence_intervals.index, confidence_intervals.iloc[:, 0], confidence_intervals.iloc[:, 1], color='green', alpha=0.2)

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data with Randomness in Future Extrapolation')
plt.legend()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 5
forecast = result.get_forecast(steps=future_steps)

# Get mean and confidence intervals for the future extrapolation
mean_values = forecast.predicted_mean
confidence_intervals = forecast.conf_int()

# Add randomness by sampling from the confidence interval
randomness = np.random.uniform(low=confidence_intervals.iloc[:, 0], high=confidence_intervals.iloc[:, 1])

# Plot original data, fitted model, and future extrapolation with randomness
plt.plot(data['TIME_SCALE'], data['COUNT'], label='Original Data')
plt.plot(result.fittedvalues.index, result.fittedvalues, color='red', label='Fitted Model')

# Plot future extrapolation with randomness
future_index = pd.RangeIndex(start=result.fittedvalues.index.max() + 1, stop=result.fittedvalues.index.max() + 1 + future_steps)
plt.plot(future_index, randomness, color='green', label='Future Extrapolation with Randomness')

# Fill the area between the confidence intervals
plt.fill_between(confidence_intervals.index, confidence_intervals.iloc[:, 0], confidence_intervals.iloc[:, 1], color='green', alpha=0.2)

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data with Randomness in Future Extrapolation')
plt.legend()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 12
forecast = result.get_forecast(steps=future_steps)

# Introduce randomness by adding white noise
# You can adjust the scale of the noise based on your preference
np.random.seed(42)  # Set seed for reproducibility
randomness = np.random.normal(scale=1.5, size=future_steps)
forecast_values_with_noise = forecast.predicted_mean + randomness

# Create a new index for the entire time range
full_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))

# Concatenate fitted values and future extrapolation values with the new index
fitted_and_forecasted = pd.concat([result.fittedvalues, forecast_values_with_noise], axis=0)
fitted_and_forecasted.index = full_index

# Plot original data, fitted model, and future extrapolation with randomness
plt.plot(data['TIME_SCALE'], data['COUNT'], label='Original Data')
plt.plot(fitted_and_forecasted.index, fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data with Randomness')
plt.legend()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 12
forecast = result.get_forecast(steps=future_steps)

# Introduce randomness by adding white noise
# You can adjust the scale of the noise based on your preference
np.random.seed(42)  # Set seed for reproducibility
randomness = np.random.normal(scale=0.1, size=future_steps)
forecast_values_with_noise = forecast.predicted_mean + randomness

# Create a new index for the entire time range
full_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))

# Concatenate fitted values and future extrapolation values with the new index
fitted_and_forecasted = pd.concat([result.fittedvalues, forecast_values_with_noise], axis=0)
fitted_and_forecasted.index = full_index

# Plot original data, fitted model, and future extrapolation with randomness
plt.figure(figsize=(10, 6))

# Plot original data
plt.plot(data['TIME_SCALE'], data['COUNT'], label='Original Data', linestyle='-', marker='o', color='blue', markersize=8)

# Plot fitted model
plt.plot(result.fittedvalues.index, result.fittedvalues, label='Fitted Model', linestyle='--', color='green')

# Plot future extrapolation with randomness
plt.plot(fitted_and_forecasted.index, fitted_and_forecasted, label='Forecast with Randomness', linestyle='-', color='red')

# Plot confidence intervals (optional)
ci = forecast.conf_int()
plt.fill_between(ci.index, ci['lower COUNT'], ci['upper COUNT'], color='pink', alpha=0.3, label='95% Confidence Interval')

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data with Randomness')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 12
forecast = result.get_forecast(steps=future_steps)

# Introduce randomness by adding white noise
# You can adjust the scale of the noise based on your preference
np.random.seed(42)  # Set seed for reproducibility
randomness = np.random.normal(scale=0.1, size=future_steps)
forecast_values_with_noise = forecast.predicted_mean + randomness

# Create a new index for the entire time range
full_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))

# Plot original data and future extrapolation with randomness
plt.figure(figsize=(10, 6))

# Plot original data
plt.plot(data['TIME_SCALE'], data['COUNT'], label='Original Data', linestyle='-', marker='o', color='blue', markersize=8)

# Plot future extrapolation with randomness
plt.plot(full_index, forecast_values_with_noise, label='Forecast with Randomness', linestyle='-', color='red')

# Plot confidence intervals (optional)
ci = forecast.conf_int()
plt.fill_between(ci.index, ci['lower COUNT'], ci['upper COUNT'], color='pink', alpha=0.3, label='95% Confidence Interval')

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data with Randomness')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 12
forecast = result.get_forecast(steps=future_steps)

# Introduce randomness by adding white noise
# You can adjust the scale of the noise based on your preference
np.random.seed(42)  # Set seed for reproducibility
randomness = np.random.normal(scale=0.1, size=future_steps)
forecast_values_with_noise = forecast.predicted_mean + randomness

# Create a new index for the entire time range
full_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))

# Ensure both x and y have the same length
data_to_plot = data['COUNT'].values
full_index_to_plot = full_index[:len(data_to_plot)]

# Plot original data and future extrapolation with randomness
plt.figure(figsize=(10, 6))

# Plot original data
plt.plot(full_index_to_plot, data_to_plot, label='Original Data', linestyle='-', marker='o', color='blue', markersize=8)

# Plot future extrapolation with randomness
plt.plot(full_index, forecast_values_with_noise, label='Forecast with Randomness', linestyle='-', color='red')

# Plot confidence intervals (optional)
ci = forecast.conf_int()
plt.fill_between(ci.index, ci['lower COUNT'], ci['upper COUNT'], color='pink', alpha=0.3, label='95% Confidence Interval')

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data with Randomness')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA

# Set Seaborn style
sns.set(style="whitegrid")

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 12
forecast = result.get_forecast(steps=future_steps)

# Introduce randomness by adding white noise
np.random.seed(42)  # Set seed for reproducibility
randomness = np.random.normal(scale=1.5, size=future_steps)
forecast_values_with_noise = forecast.predicted_mean + randomness

# Create a new index for the entire time range
full_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))

# Concatenate fitted values and future extrapolation values with the new index
fitted_and_forecasted = pd.concat([result.fittedvalues, forecast_values_with_noise], axis=0)
fitted_and_forecasted.index = full_index

# Use Seaborn's lineplot for better handling of lines and markers
sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data with Randomness')
plt.legend()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA

# Set Seaborn style
sns.set(style="whitegrid")

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 12
forecast = result.get_forecast(steps=future_steps)

# Introduce randomness by adding white noise
np.random.seed(42)  # Set seed for reproducibility
randomness = np.random.normal(scale=1.5, size=future_steps)
forecast_values_with_noise = forecast.predicted_mean + randomness

# Create a new index for the entire time range
full_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))

# Concatenate fitted values and future extrapolation values with the new index
fitted_and_forecasted = pd.concat([result.fittedvalues, forecast_values_with_noise], axis=0)
fitted_and_forecasted.index = full_index

# Set a larger figure size
plt.figure(figsize=(10, 6))

# Use Seaborn's lineplot for better handling of lines and markers
sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data with Randomness')
plt.legend()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA

# Set Seaborn style
sns.set(style="whitegrid")

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 12
forecast = result.get_forecast(steps=future_steps)

# Extract upper and lower bounds of the confidence interval
ci = forecast.conf_int()

# Introduce randomness by adding white noise
np.random.seed(42)  # Set seed for reproducibility
randomness = np.random.normal(scale=1.5, size=future_steps)
forecast_values_with_noise = forecast.predicted_mean + randomness

# Create a new index for the entire time range
full_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))

# Concatenate fitted values and future extrapolation values with the new index
fitted_and_forecasted = pd.concat([result.fittedvalues, forecast_values_with_noise], axis=0)
fitted_and_forecasted.index = full_index

# Set a larger figure size
plt.figure(figsize=(10, 6))

# Use Seaborn's lineplot for better handling of lines and markers
sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')
plt.fill_between(fitted_and_forecasted.index, ci.iloc[:, 0], ci.iloc[:, 1], color='red', alpha=0.2, label='95% Confidence Interval')

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data with Randomness')
plt.legend()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA

# Set Seaborn style
sns.set(style="whitegrid")

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 12
forecast = result.get_forecast(steps=future_steps)

# Extract upper and lower bounds of the confidence interval
ci = forecast.conf_int()

# Introduce randomness by adding white noise
np.random.seed(42)  # Set seed for reproducibility
randomness = np.random.normal(scale=1.5, size=future_steps)
forecast_values_with_noise = forecast.predicted_mean + randomness

# Use the common index between forecast values and confidence interval
common_index = forecast_values_with_noise.index

# Set a larger figure size
plt.figure(figsize=(10, 6))

# Use Seaborn's lineplot for better handling of lines and markers
sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')
plt.fill_between(common_index, ci.iloc[:, 0], ci.iloc[:, 1], color='red', alpha=0.2, label='95% Confidence Interval')

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data with Randomness')
plt.legend()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA

# Set Seaborn style
sns.set(style="whitegrid")

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 12
forecast = result.get_forecast(steps=future_steps)

# Extract upper and lower bounds of the confidence interval
ci = forecast.conf_int()

# Introduce randomness by adding white noise
np.random.seed(42)  # Set seed for reproducibility
randomness = np.random.normal(scale=1.5, size=future_steps)
forecast_values_with_noise = forecast.predicted_mean + randomness

# Use the common index between forecast values and confidence interval
common_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))

# Concatenate fitted values and future extrapolation values with the new index
fitted_and_forecasted = pd.concat([result.fittedvalues, forecast_values_with_noise], axis=0)
fitted_and_forecasted.index = common_index

# Set a larger figure size
plt.figure(figsize=(10, 6))

# Use Seaborn's lineplot for better handling of lines and markers
sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')
plt.fill_between(common_index, ci.iloc[:, 0], ci.iloc[:, 1], color='red', alpha=0.2, label='95% Confidence Interval')

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data with Randomness')
plt.legend()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA

# Set Seaborn style
sns.set(style="whitegrid")

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 12
forecast = result.get_forecast(steps=future_steps)

# Extract upper and lower bounds of the confidence interval
ci = forecast.conf_int()

# Introduce randomness by adding white noise
np.random.seed(42)  # Set seed for reproducibility
randomness = np.random.normal(scale=1.5, size=future_steps)
forecast_values_with_noise = forecast.predicted_mean + randomness

# Use the common index between forecast values and confidence interval
common_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))

# Truncate common_index to match the length of forecast_values_with_noise
common_index_truncated = common_index[:len(forecast_values_with_noise)]

# Set a larger figure size
plt.figure(figsize=(10, 6))

# Use Seaborn's lineplot for better handling of lines and markers
sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')
plt.fill_between(common_index_truncated, ci.iloc[:, 0], ci.iloc[:, 1], color='red', alpha=0.2, label='95% Confidence Interval')

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data with Randomness')
plt.legend()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA

# Set Seaborn style
sns.set(style="whitegrid")

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 12
forecast = result.get_forecast(steps=future_steps)

# Extract upper and lower bounds of the confidence interval
ci = forecast.conf_int()

# Introduce randomness by adding white noise
np.random.seed(42)  # Set seed for reproducibility
randomness = np.random.normal(scale=1.5, size=future_steps)
forecast_values_with_noise = forecast.predicted_mean + randomness

# Set a larger figure size
plt.figure(figsize=(10, 6))

# Use Seaborn's lineplot for better handling of lines and markers
sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')

# Use directly the index of fitted_and_forecasted for the confidence interval
plt.fill_between(fitted_and_forecasted.index, ci.iloc[:, 0], ci.iloc[:, 1], color='red', alpha=0.2, label='95% Confidence Interval')

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data with Randomness')
plt.legend()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA

# Set Seaborn style
sns.set(style="whitegrid")

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 12
forecast = result.get_forecast(steps=future_steps)

# Extract upper and lower bounds of the confidence interval
ci = forecast.conf_int()

# Introduce randomness by adding white noise
np.random.seed(42)  # Set seed for reproducibility
randomness = np.random.normal(scale=1.5, size=future_steps)
forecast_values_with_noise = forecast.predicted_mean + randomness

# Use the common index between forecast values and confidence interval
common_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))

# Truncate common_index to match the length of forecast_values_with_noise
common_index_truncated = common_index[:len(forecast_values_with_noise)]

# Interpolate confidence interval to match the length of fitted and forecasted values
ci_interpolated_lower = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 0])
ci_interpolated_upper = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 1])

# Set a larger figure size
plt.figure(figsize=(10, 6))

# Use Seaborn's lineplot for better handling of lines and markers
sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')

# Use interpolated confidence interval
plt.fill_between(fitted_and_forecasted.index, ci_interpolated_lower, ci_interpolated_upper, color='red', alpha=0.2, label='95% Confidence Interval')

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data with Randomness')
plt.legend()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA

# Set Seaborn style
sns.set(style="whitegrid")

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 12
forecast = result.get_forecast(steps=future_steps)

# Extract upper and lower bounds of the confidence interval
ci = forecast.conf_int()

# Introduce randomness by adding white noise
np.random.seed(42)  # Set seed for reproducibility
randomness = np.random.normal(scale=1.5, size=future_steps)
forecast_values_with_noise = forecast.predicted_mean + randomness

# Use the common index between forecast values and confidence interval
common_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))

# Truncate common_index to match the length of forecast_values_with_noise
common_index_truncated = common_index[:len(forecast_values_with_noise)]

# Interpolate confidence interval to match the length of fitted and forecasted values
ci_interpolated_lower = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 0])
ci_interpolated_upper = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 1])

# Set a larger figure size
plt.figure(figsize=(10, 6))

# Use Seaborn's lineplot for better handling of lines and markers
sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')

# Use interpolated confidence interval with reduced alpha
plt.fill_between(fitted_and_forecasted.index, ci_interpolated_lower, ci_interpolated_upper, color='red', alpha=0.1, label='95% Confidence Interval')

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data with Randomness')
plt.legend()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA

# Set Seaborn style
sns.set(style="whitegrid")

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 12
forecast = result.get_forecast(steps=future_steps)

# Extract upper and lower bounds of the confidence interval
ci = forecast.conf_int(alpha=0.25)  # Set alpha to 0.25 for 25% confidence interval

# Introduce randomness by adding white noise
np.random.seed(42)  # Set seed for reproducibility
randomness = np.random.normal(scale=1.5, size=future_steps)
forecast_values_with_noise = forecast.predicted_mean + randomness

# Use the common index between forecast values and confidence interval
common_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))

# Truncate common_index to match the length of forecast_values_with_noise
common_index_truncated = common_index[:len(forecast_values_with_noise)]

# Interpolate confidence interval to match the length of fitted and forecasted values
ci_interpolated_lower = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 0])
ci_interpolated_upper = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 1])

# Set a larger figure size
plt.figure(figsize=(10, 6))

# Use Seaborn's lineplot for better handling of lines and markers
sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')

# Use interpolated confidence interval with reduced alpha
plt.fill_between(fitted_and_forecasted.index, ci_interpolated_lower, ci_interpolated_upper, color='red', alpha=0.2, label='25% Confidence Interval')

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data with Randomness')
plt.legend()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA

# Set Seaborn style
sns.set(style="whitegrid")

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 12
forecast = result.get_forecast(steps=future_steps)

# Extract upper and lower bounds of the confidence interval
ci = forecast.conf_int(alpha=0.1)  # Set alpha to 0.1 for 10% confidence interval

# Introduce randomness by adding white noise
np.random.seed(42)  # Set seed for reproducibility
randomness = np.random.normal(scale=1.5, size=future_steps)
forecast_values_with_noise = forecast.predicted_mean + randomness

# Use the common index between forecast values and confidence interval
common_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))

# Truncate common_index to match the length of forecast_values_with_noise
common_index_truncated = common_index[:len(forecast_values_with_noise)]

# Interpolate confidence interval to match the length of fitted and forecasted values
ci_interpolated_lower = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 0])
ci_interpolated_upper = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 1])

# Set a larger figure size
plt.figure(figsize=(10, 6))

# Use Seaborn's lineplot for better handling of lines and markers
sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')

# Use interpolated confidence interval with reduced alpha
plt.fill_between(fitted_and_forecasted.index, ci_interpolated_lower, ci_interpolated_upper, color='red', alpha=0.2, label='10% Confidence Interval')

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data with Randomness')
plt.legend()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA

# Set Seaborn style
sns.set(style="whitegrid")

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 12
forecast = result.get_forecast(steps=future_steps)

# Extract upper and lower bounds of the confidence interval
ci = forecast.conf_int(alpha=0.25)  # Set alpha to 0.25 for 25% confidence interval

# Introduce randomness by adding white noise
np.random.seed(42)  # Set seed for reproducibility
randomness = np.random.normal(scale=1.5, size=future_steps)
forecast_values_with_noise = forecast.predicted_mean + randomness

# Use the common index between forecast values and confidence interval
common_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))

# Truncate common_index to match the length of forecast_values_with_noise
common_index_truncated = common_index[:len(forecast_values_with_noise)]

# Interpolate confidence interval to match the length of fitted and forecasted values
ci_interpolated_lower = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 0])
ci_interpolated_upper = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 1])

# Set a larger figure size
plt.figure(figsize=(10, 6))

# Use Seaborn's lineplot for better handling of lines and markers
sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')

# Use interpolated confidence interval with reduced alpha
plt.fill_between(fitted_and_forecasted.index, ci_interpolated_lower, ci_interpolated_upper, color='red', alpha=0.2, label='25% Confidence Interval')

# Set y-axis to be logarithmic and show only positive values
plt.yscale('log')
plt.yticks([1, 10, 100, 1000], ['1', '10', '100', '1000'])

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data with Randomness')
plt.legend()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA

# Set Seaborn style
sns.set(style="whitegrid")

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 12
forecast = result.get_forecast(steps=future_steps)

# Extract upper and lower bounds of the confidence interval
ci = forecast.conf_int(alpha=0.85)  # Set alpha to 0.25 for 25% confidence interval

# Introduce randomness by adding white noise
np.random.seed(42)  # Set seed for reproducibility
randomness = np.random.normal(scale=1.5, size=future_steps)
forecast_values_with_noise = forecast.predicted_mean + randomness

# Use the common index between forecast values and confidence interval
common_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))

# Truncate common_index to match the length of forecast_values_with_noise
common_index_truncated = common_index[:len(forecast_values_with_noise)]

# Interpolate confidence interval to match the length of fitted and forecasted values
ci_interpolated_lower = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 0])
ci_interpolated_upper = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 1])

# Set a larger figure size
plt.figure(figsize=(10, 6))

# Use Seaborn's lineplot for better handling of lines and markers
sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')

# Use interpolated confidence interval with reduced alpha
plt.fill_between(fitted_and_forecasted.index, ci_interpolated_lower, ci_interpolated_upper, color='red', alpha=0.2, label='85% Confidence Interval')

# Set y-axis lower limit to 0
plt.ylim(bottom=0)

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data with Randomness')
plt.legend()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA

# Set Seaborn style
sns.set(style="whitegrid")

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 12
forecast = result.get_forecast(steps=future_steps)

# Extract upper and lower bounds of the confidence interval
ci = forecast.conf_int(alpha=0.85)  # Set alpha to 0.85 for 85% confidence interval

# Introduce randomness by adding white noise
np.random.seed(42)  # Set seed for reproducibility
randomness = np.random.normal(scale=1.5, size=future_steps)
forecast_values_with_noise = forecast.predicted_mean + randomness

# Use the common index between forecast values and confidence interval
common_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))

# Truncate common_index to match the length of forecast_values_with_noise
common_index_truncated = common_index[:len(forecast_values_with_noise)]

# Interpolate confidence interval to match the length of fitted and forecasted values
ci_interpolated_lower = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 0])
ci_interpolated_upper = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 1])

# Set a larger figure size
plt.figure(figsize=(10, 6))

# Use Seaborn's lineplot for better handling of lines and markers
sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')

# Use interpolated confidence interval with reduced alpha
plt.fill_between(fitted_and_forecasted.index, ci_interpolated_lower, ci_interpolated_upper, color='red', alpha=0.2, label='85% Confidence Interval')

# Set y-axis lower limit to 0
plt.ylim(bottom=0)

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data with Randomness')
plt.legend()
plt.show()

# Compute and print the covariance matrix, standard deviation, and MAE
summary = result.summary()
cov_matrix = result.cov_params()
std_dev = result.bse
mae = np.mean(np.abs(data['COUNT'] - result.fittedvalues))
print("\nCovariance Matrix:")
print(cov_matrix)
print("\nStandard Deviation:")
print(std_dev)
print("\nMean Absolute Error (MAE):", mae)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from tabulate import tabulate  # Import tabulate from the tabulate library

# Set Seaborn style
sns.set(style="whitegrid")

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 12
forecast = result.get_forecast(steps=future_steps)

# Extract upper and lower bounds of the confidence interval
ci = forecast.conf_int(alpha=0.85)  # Set alpha to 0.85 for 85% confidence interval

# Introduce randomness by adding white noise
np.random.seed(42)  # Set seed for reproducibility
randomness = np.random.normal(scale=1.5, size=future_steps)
forecast_values_with_noise = forecast.predicted_mean + randomness

# Use the common index between forecast values and confidence interval
common_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))

# Truncate common_index to match the length of forecast_values_with_noise
common_index_truncated = common_index[:len(forecast_values_with_noise)]

# Interpolate confidence interval to match the length of fitted and forecasted values
ci_interpolated_lower = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 0])
ci_interpolated_upper = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 1])

# Set a larger figure size
plt.figure(figsize=(10, 6))

# Use Seaborn's lineplot for better handling of lines and markers
sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')

# Use interpolated confidence interval with reduced alpha
plt.fill_between(fitted_and_forecasted.index, ci_interpolated_lower, ci_interpolated_upper, color='red', alpha=0.2, label='85% Confidence Interval')

# Set y-axis lower limit to 0
plt.ylim(bottom=0)

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data with Randomness')
plt.legend()
plt.show()

# Compute and print the covariance matrix, standard deviation, and MAE
summary = result.summary()
cov_matrix = result.cov_params()
std_dev = result.bse
mae = np.mean(np.abs(data['COUNT'] - result.fittedvalues))

# Create a beautiful table
table_data = {
    'Covariance Matrix': cov_matrix,
    'Standard Deviation': std_dev,
    'Mean Absolute Error (MAE)': mae
}

table = tabulate(table_data, headers='keys', tablefmt='pretty')
print(table)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from tabulate import tabulate  # Import tabulate from the tabulate library

# Set Seaborn style
sns.set(style="whitegrid")

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 12
forecast = result.get_forecast(steps=future_steps)

# Extract upper and lower bounds of the confidence interval
ci = forecast.conf_int(alpha=0.85)  # Set alpha to 0.85 for 85% confidence interval

# Introduce randomness by adding white noise
np.random.seed(42)  # Set seed for reproducibility
randomness = np.random.normal(scale=1.5, size=future_steps)
forecast_values_with_noise = forecast.predicted_mean + randomness

# Use the common index between forecast values and confidence interval
common_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))

# Truncate common_index to match the length of forecast_values_with_noise
common_index_truncated = common_index[:len(forecast_values_with_noise)]

# Interpolate confidence interval to match the length of fitted and forecasted values
ci_interpolated_lower = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 0])
ci_interpolated_upper = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 1])

# Set a larger figure size
plt.figure(figsize=(10, 6))

# Use Seaborn's lineplot for better handling of lines and markers
sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')

# Use interpolated confidence interval with reduced alpha
plt.fill_between(fitted_and_forecasted.index, ci_interpolated_lower, ci_interpolated_upper, color='red', alpha=0.2, label='85% Confidence Interval')

# Set y-axis lower limit to 0
plt.ylim(bottom=0)

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data with Randomness')
plt.legend()
plt.show()

# Compute and print the covariance matrix, standard deviation, and MAE
summary = result.summary()
cov_matrix = pd.DataFrame(result.cov_params(), columns=result.params.index)
std_dev = pd.DataFrame(result.bse, columns=['Standard Deviation'])
mae = np.mean(np.abs(data['COUNT'] - result.fittedvalues))

# Create a beautiful table
table_data = {
    'Covariance Matrix': cov_matrix,
    'Standard Deviation': std_dev,
    'Mean Absolute Error (MAE)': mae
}

table = tabulate(table_data, headers='keys', tablefmt='pretty')
print(table)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from tabulate import tabulate  # Import tabulate from the tabulate library

# Set Seaborn style
sns.set(style="whitegrid")

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 12
forecast = result.get_forecast(steps=future_steps)

# Extract upper and lower bounds of the confidence interval
ci = forecast.conf_int(alpha=0.85)  # Set alpha to 0.85 for 85% confidence interval

# Introduce randomness by adding white noise
np.random.seed(42)  # Set seed for reproducibility
randomness = np.random.normal(scale=1.5, size=future_steps)
forecast_values_with_noise = forecast.predicted_mean + randomness

# Use the common index between forecast values and confidence interval
common_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))

# Truncate common_index to match the length of forecast_values_with_noise
common_index_truncated = common_index[:len(forecast_values_with_noise)]

# Interpolate confidence interval to match the length of fitted and forecasted values
ci_interpolated_lower = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 0])
ci_interpolated_upper = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 1])

# Set a larger figure size
plt.figure(figsize=(10, 6))

# Use Seaborn's lineplot for better handling of lines and markers
sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')

# Use interpolated confidence interval with reduced alpha
plt.fill_between(fitted_and_forecasted.index, ci_interpolated_lower, ci_interpolated_upper, color='red', alpha=0.2, label='85% Confidence Interval')

# Set y-axis lower limit to 0
plt.ylim(bottom=0)

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data with Randomness')
plt.legend()
plt.show()

# Compute and print the covariance matrix, standard deviation, and MAE
summary = result.summary()
cov_matrix = pd.DataFrame(result.cov_params(), columns=result.params.index)
std_dev = pd.DataFrame(result.bse, columns=['Standard Deviation'])
mae = np.mean(np.abs(data['COUNT'] - result.fittedvalues))

# Convert covariance matrix to a list of lists
cov_matrix_list = cov_matrix.values.tolist()

# Create a beautiful table
table_data = {
    'Covariance Matrix': cov_matrix_list,
    'Standard Deviation': std_dev,
    'Mean Absolute Error (MAE)': mae
}

table = tabulate(table_data, headers='keys', tablefmt='pretty')
print(table)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from tabulate import tabulate  # Import tabulate from the tabulate library

# Set Seaborn style
sns.set(style="whitegrid")

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 12
forecast = result.get_forecast(steps=future_steps)

# Extract upper and lower bounds of the confidence interval
ci = forecast.conf_int(alpha=0.85)  # Set alpha to 0.85 for 85% confidence interval

# Introduce randomness by adding white noise
np.random.seed(42)  # Set seed for reproducibility
randomness = np.random.normal(scale=1.5, size=future_steps)
forecast_values_with_noise = forecast.predicted_mean + randomness

# Use the common index between forecast values and confidence interval
common_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))

# Truncate common_index to match the length of forecast_values_with_noise
common_index_truncated = common_index[:len(forecast_values_with_noise)]

# Interpolate confidence interval to match the length of fitted and forecasted values
ci_interpolated_lower = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 0])
ci_interpolated_upper = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 1])

# Set a larger figure size
plt.figure(figsize=(10, 6))

# Use Seaborn's lineplot for better handling of lines and markers
sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')

# Use interpolated confidence interval with reduced alpha
plt.fill_between(fitted_and_forecasted.index, ci_interpolated_lower, ci_interpolated_upper, color='red', alpha=0.2, label='85% Confidence Interval')

# Set y-axis lower limit to 0
plt.ylim(bottom=0)

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data with Randomness')
plt.legend()
plt.show()

# Compute and print the covariance matrix, standard deviation, and MAE
summary = result.summary()
cov_matrix = pd.DataFrame(result.cov_params(), columns=result.params.index)
std_dev = pd.DataFrame(result.bse, columns=['Standard Deviation'])
mae = np.mean(np.abs(data['COUNT'] - result.fittedvalues))

# Convert covariance matrix to a list of lists
cov_matrix_list = cov_matrix.applymap(lambda x: float(x)).values.tolist()

# Create a beautiful table
table_data = {
    'Covariance Matrix': cov_matrix_list,
    'Standard Deviation': std_dev,
    'Mean Absolute Error (MAE)': mae
}

table = tabulate(table_data, headers='keys', tablefmt='pretty')
print(table)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from tabulate import tabulate  # Import tabulate from the tabulate library

# Set Seaborn style
sns.set(style="whitegrid")

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 12
forecast = result.get_forecast(steps=future_steps)

# Extract upper and lower bounds of the confidence interval
ci = forecast.conf_int(alpha=0.85)  # Set alpha to 0.85 for 85% confidence interval

# Introduce randomness by adding white noise
np.random.seed(42)  # Set seed for reproducibility
randomness = np.random.normal(scale=1.5, size=future_steps)
forecast_values_with_noise = forecast.predicted_mean + randomness

# Use the common index between forecast values and confidence interval
common_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))

# Truncate common_index to match the length of forecast_values_with_noise
common_index_truncated = common_index[:len(forecast_values_with_noise)]

# Interpolate confidence interval to match the length of fitted and forecasted values
ci_interpolated_lower = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 0])
ci_interpolated_upper = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 1])

# Set a larger figure size
plt.figure(figsize=(10, 6))

# Use Seaborn's lineplot for better handling of lines and markers
sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')

# Use interpolated confidence interval with reduced alpha
plt.fill_between(fitted_and_forecasted.index, ci_interpolated_lower, ci_interpolated_upper, color='red', alpha=0.2, label='85% Confidence Interval')

# Set y-axis lower limit to 0
plt.ylim(bottom=0)

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data with Randomness')
plt.legend()
plt.show()

# Compute and print the covariance matrix, standard deviation, and MAE
summary = result.summary()
cov_matrix = pd.DataFrame(result.cov_params(), columns=result.params.index)
std_dev = pd.DataFrame(result.bse, columns=['Standard Deviation'])
mae = np.mean(np.abs(data['COUNT'] - result.fittedvalues))

# Convert covariance matrix to a list of lists
cov_matrix_list = cov_matrix.applymap(lambda x: f"{x:.4f}").values.tolist()
std_dev_list = std_dev.applymap(lambda x: f"{x:.4f}").values.tolist()

# Create a beautiful table
table_data = {
    'Covariance Matrix': cov_matrix_list,
    'Standard Deviation': std_dev_list,
    'Mean Absolute Error (MAE)': [f"{mae:.4f}"]
}

table = tabulate(table_data, headers='keys', tablefmt='pretty')
print(table)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 12
forecast = result.get_forecast(steps=future_steps)

# True values (actual data)
true_values = data['COUNT']

# Extract upper and lower bounds of the confidence interval
ci = forecast.conf_int(alpha=0.85)  # Set alpha to 0.85 for 85% confidence interval

# Introduce randomness by adding white noise
np.random.seed(42)  # Set seed for reproducibility
randomness = np.random.normal(scale=1.5, size=future_steps)
forecast_values_with_noise = forecast.predicted_mean + randomness

# Calculate Mean Squared Error (MSE) for each time point
mse_values = np.square(true_values - forecast_values_with_noise).mean(axis=1)

# Plot the overall trend of MSE
plt.figure(figsize=(10, 6))
sns.lineplot(x=common_index_truncated, y=mse_values, label='Mean Squared Error (MSE)', marker='o', color='green')
plt.xlabel('Time Scale')
plt.ylabel('MSE')
plt.title('Overall Trend of Mean Squared Error (MSE) over Time')
plt.legend()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from tabulate import tabulate

class ARIMAModel:
    def __init__(self, data):
        self.data = data
        self.p, self.d, self.q = 1, 1, 1  # Example values for p, d, q
        self.order = (self.p, self.d, self.q)
        self.model = ARIMA(self.data['COUNT'], order=self.order)
        self.result = self.model.fit()

    def forecast(self, future_steps):
        forecast = self.result.get_forecast(steps=future_steps)
        return forecast

    def introduce_randomness(self, forecast_values, future_steps):
        np.random.seed(42)  # Set seed for reproducibility
        randomness = np.random.normal(scale=1.5, size=future_steps)
        forecast_values_with_noise = forecast_values.predicted_mean + randomness
        return forecast_values_with_noise

    def plot_forecast(self, fitted_and_forecasted, ci_interpolated_lower, ci_interpolated_upper):
        common_index = pd.Index(list(self.data['TIME_SCALE']) + list(range(self.data['TIME_SCALE'].max() + 1, self.data['TIME_SCALE'].max() + 1 + len(fitted_and_forecasted))))
        common_index_truncated = common_index[:len(fitted_and_forecasted)]

        plt.figure(figsize=(10, 6))
        sns.lineplot(x=self.data['TIME_SCALE'], y=self.data['COUNT'], label='Original Data')
        sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')
        plt.fill_between(fitted_and_forecasted.index, ci_interpolated_lower, ci_interpolated_upper, color='red', alpha=0.2, label='85% Confidence Interval')
        plt.ylim(bottom=0)
        plt.xlabel('Time Scale')
        plt.ylabel('Count')
        plt.title('ARIMA Model for Time Series Data with Randomness')
        plt.legend()
        plt.show()

    def get_summary_statistics(self):
        summary = self.result.summary()
        cov_matrix = pd.DataFrame(self.result.cov_params(), columns=self.result.params.index)
        std_dev = pd.DataFrame(self.result.bse, columns=['Standard Deviation'])
        mae = np.mean(np.abs(self.data['COUNT'] - self.result.fittedvalues))

        cov_matrix_list = cov_matrix.applymap(lambda x: f"{x:.4f}").values.tolist()
        std_dev_list = std_dev.applymap(lambda x: f"{x:.4f}").values.tolist()

        return cov_matrix_list, std_dev_list, mae

    def print_statistics_table(self, cov_matrix_list, std_dev_list, mae):
        table_data = {
            'Covariance Matrix': cov_matrix_list,
            'Standard Deviation': std_dev_list,
            'Mean Absolute Error (MAE)': [f"{mae:.4f}"]
        }

        table = tabulate(table_data, headers='keys', tablefmt='pretty')
        print(table)


# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame
arima_model = ARIMAModel(data)

# Number of future steps to extrapolate
future_steps = 12
forecast_values = arima_model.forecast(future_steps)

# Introduce randomness
forecast_values_with_noise = arima_model.introduce_randomness(forecast_values, future_steps)

# Plot the forecast
arima_model.plot_forecast(forecast_values_with_noise, ci_interpolated_lower, ci_interpolated_upper)

# Get and print summary statistics
cov_matrix_list, std_dev_list, mae = arima_model.get_summary_statistics()
arima_model.print_statistics_table(cov_matrix_list, std_dev_list, mae)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from tabulate import tabulate  # Import tabulate from the tabulate library

# Set Seaborn style
sns.set(style="whitegrid")

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 12
forecast = result.get_forecast(steps=future_steps)

# Extract upper and lower bounds of the confidence interval
ci = forecast.conf_int(alpha=0.85)  # Set alpha to 0.85 for 85% confidence interval

# Introduce randomness by adding white noise
np.random.seed(42)  # Set seed for reproducibility
randomness = np.random.normal(scale=1.5, size=future_steps)
forecast_values_with_noise = forecast.predicted_mean + randomness

# Use the common index between forecast values and confidence interval
common_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))

# Truncate common_index to match the length of forecast_values_with_noise
common_index_truncated = common_index[:len(forecast_values_with_noise)]

# Interpolate confidence interval to match the length of fitted and forecasted values
ci_interpolated_lower = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 0])
ci_interpolated_upper = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 1])

# Set a larger figure size
plt.figure(figsize=(10, 6))

# Use Seaborn's lineplot for better handling of lines and markers
sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')

# Use interpolated confidence interval with reduced alpha
plt.fill_between(fitted_and_forecasted.index, ci_interpolated_lower, ci_interpolated_upper, color='red', alpha=0.2, label='85% Confidence Interval')

# Set y-axis lower limit to 0
plt.ylim(bottom=0)

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data with Randomness')
plt.legend()
plt.show()

# Compute and print the covariance matrix, standard deviation, and MAE
summary = result.summary()
cov_matrix = pd.DataFrame(result.cov_params(), columns=result.params.index)
std_dev = pd.DataFrame(result.bse, columns=['Standard Deviation'])
mae = np.mean(np.abs(data['COUNT'] - result.fittedvalues))

# Convert covariance matrix to a list of lists
cov_matrix_list = cov_matrix.applymap(lambda x: f"{x:.4f}").values.tolist()
std_dev_list = std_dev.applymap(lambda x: f"{x:.4f}").values.tolist()

# Create a beautiful table
table_data = {
    'Covariance Matrix': cov_matrix_list,
    'Standard Deviation': std_dev_list,
    'Mean Absolute Error (MAE)': [f"{mae:.4f}"]
}

table = tabulate(table_data, headers='keys', tablefmt='pretty')
print(table)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from tabulate import tabulate

class ARIMAModel:
    def __init__(self, data, p=1, d=1, q=1, alpha=0.85, future_steps=12, randomness_seed=42, noise_scale=1.5):
        self.data = data
        self.p, self.d, self.q = p, d, q
        self.alpha = alpha
        self.future_steps = future_steps
        self.randomness_seed = randomness_seed
        self.noise_scale = noise_scale
        self.order = (p, d, q)

    def fit_arima_model(self):
        model = ARIMA(self.data['COUNT'], order=self.order)
        result = model.fit()
        return result

    def forecast_future_steps(self, result):
        forecast = result.get_forecast(steps=self.future_steps)
        return forecast

    def introduce_randomness(self, forecast):
        np.random.seed(self.randomness_seed)
        randomness = np.random.normal(scale=self.noise_scale, size=self.future_steps)
        forecast_values_with_noise = forecast.predicted_mean + randomness
        return forecast_values_with_noise

    def create_common_index(self):
        common_index = pd.Index(list(self.data['TIME_SCALE']) + list(range(self.data['TIME_SCALE'].max() + 1, self.data['TIME_SCALE'].max() + 1 + self.future_steps)))
        return common_index

    def interpolate_confidence_interval(self, forecast_values_with_noise, common_index):
        ci = self.forecast.conf_int(alpha=self.alpha)
        common_index_truncated = common_index[:len(forecast_values_with_noise)]
        ci_interpolated_lower = np.interp(common_index_truncated, common_index, ci.iloc[:, 0])
        ci_interpolated_upper = np.interp(common_index_truncated, common_index, ci.iloc[:, 1])
        return ci_interpolated_lower, ci_interpolated_upper

def plot_arima_results(data, fitted_and_forecasted, ci_interpolated_lower, ci_interpolated_upper):
    plt.figure(figsize=(10, 6))
    sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
    sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')
    plt.fill_between(fitted_and_forecasted.index, ci_interpolated_lower, ci_interpolated_upper, color='red', alpha=0.2, label='85% Confidence Interval')
    plt.ylim(bottom=0)
    plt.xlabel('Time Scale')
    plt.ylabel('Count')
    plt.title('ARIMA Model for Time Series Data with Randomness')
    plt.legend()
    plt.show()

def print_arima_summary(result, data):
    summary = result.summary()
    cov_matrix = pd.DataFrame(result.cov_params(), columns=result.params.index)
    std_dev = pd.DataFrame(result.bse, columns=['Standard Deviation'])
    mae = np.mean(np.abs(data['COUNT'] - result.fittedvalues))

    cov_matrix_list = cov_matrix.applymap(lambda x: f"{x:.4f}").values.tolist()
    std_dev_list = std_dev.applymap(lambda x: f"{x:.4f}").values.tolist()

    table_data = {
        'Covariance Matrix': cov_matrix_list,
        'Standard Deviation': std_dev_list,
        'Mean Absolute Error (MAE)': [f"{mae:.4f}"]
    }

    table = tabulate(table_data, headers='keys', tablefmt='pretty')
    print(table)

# Example usage:
arima_model = ARIMAModel(data)
result = arima_model.fit_arima_model()
forecast = arima_model.forecast_future_steps(result)
forecast_values_with_noise = arima_model.introduce_randomness(forecast)
common_index = arima_model.create_common_index()
ci_interpolated_lower, ci_interpolated_upper = arima_model.interpolate_confidence_interval(forecast_values_with_noise, common_index)

plot_arima_results(data, forecast_values_with_noise, ci_interpolated_lower, ci_interpolated_upper)
print_arima_summary(result, data)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from tabulate import tabulate

class ARIMAModel:
    def __init__(self, data, p=1, d=1, q=1, alpha=0.85, future_steps=12, randomness_seed=42, noise_scale=1.5):
        self.data = data
        self.p, self.d, self.q = p, d, q
        self.alpha = alpha
        self.future_steps = future_steps
        self.randomness_seed = randomness_seed
        self.noise_scale = noise_scale
        self.order = (p, d, q)

    def fit_arima_model(self):
        model = ARIMA(self.data['COUNT'], order=self.order)
        result = model.fit()
        return result

    def forecast_future_steps(self, result):
        forecast = result.get_forecast(steps=self.future_steps)
        return forecast

    def introduce_randomness(self, forecast):
        np.random.seed(self.randomness_seed)
        randomness = np.random.normal(scale=self.noise_scale, size=self.future_steps)
        forecast_values_with_noise = forecast.predicted_mean + randomness
        return forecast_values_with_noise

    def create_common_index(self):
        common_index = pd.Index(list(self.data['TIME_SCALE']) + list(range(self.data['TIME_SCALE'].max() + 1, self.data['TIME_SCALE'].max() + 1 + self.future_steps)))
        return common_index

    def interpolate_confidence_interval(self, forecast, common_index):
        ci = forecast.conf_int(alpha=self.alpha)
        common_index_truncated = common_index[:len(forecast.predicted_mean)]
        ci_interpolated_lower = np.interp(common_index_truncated, common_index, ci.iloc[:, 0])
        ci_interpolated_upper = np.interp(common_index_truncated, common_index, ci.iloc[:, 1])
        return ci_interpolated_lower, ci_interpolated_upper

def plot_arima_results(data, fitted_and_forecasted, ci_interpolated_lower, ci_interpolated_upper):
    plt.figure(figsize=(10, 6))
    sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
    sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')
    plt.fill_between(fitted_and_forecasted.index, ci_interpolated_lower, ci_interpolated_upper, color='red', alpha=0.2, label='85% Confidence Interval')
    plt.ylim(bottom=0)
    plt.xlabel('Time Scale')
    plt.ylabel('Count')
    plt.title('ARIMA Model for Time Series Data with Randomness')
    plt.legend()
    plt.show()

def print_arima_summary(result, data):
    summary = result.summary()
    cov_matrix = pd.DataFrame(result.cov_params(), columns=result.params.index)
    std_dev = pd.DataFrame(result.bse, columns=['Standard Deviation'])
    mae = np.mean(np.abs(data['COUNT'] - result.fittedvalues))

    cov_matrix_list = cov_matrix.applymap(lambda x: f"{x:.4f}").values.tolist()
    std_dev_list = std_dev.applymap(lambda x: f"{x:.4f}").values.tolist()

    table_data = {
        'Covariance Matrix': cov_matrix_list,
        'Standard Deviation': std_dev_list,
        'Mean Absolute Error (MAE)': [f"{mae:.4f}"]
    }

    table = tabulate(table_data, headers='keys', tablefmt='pretty')
    print(table)

# Example usage:
arima_model = ARIMAModel(data)
result = arima_model.fit_arima_model()
forecast = arima_model.forecast_future_steps(result)
forecast_values_with_noise = arima_model.introduce_randomness(forecast)
common_index = arima_model.create_common_index()
ci_interpolated_lower, ci_interpolated_upper = arima_model.interpolate_confidence_interval(forecast, common_index)

plot_arima_results(data, forecast_values_with_noise, ci_interpolated_lower, ci_interpolated_upper)
print_arima_summary(result, data)


In [ ]:
import pandas as pd 
    
    # Load time series data from CSV file
file_path = "/Users/igormol/Desktop/John/0105010006.csv"
data = pd.read_csv(file_path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from tabulate import tabulate  # Import tabulate from the tabulate library

# Set Seaborn style
sns.set(style="whitegrid")

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 12
forecast = result.get_forecast(steps=future_steps)

# Extract upper and lower bounds of the confidence interval
ci = forecast.conf_int(alpha=0.85)  # Set alpha to 0.85 for 85% confidence interval

# Introduce randomness by adding white noise
np.random.seed(42)  # Set seed for reproducibility
randomness = np.random.normal(scale=1.5, size=future_steps)
forecast_values_with_noise = forecast.predicted_mean + randomness

# Use the common index between forecast values and confidence interval
common_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))

# Truncate common_index to match the length of forecast_values_with_noise
common_index_truncated = common_index[:len(forecast_values_with_noise)]

# Interpolate confidence interval to match the length of fitted and forecasted values
ci_interpolated_lower = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 0])
ci_interpolated_upper = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 1])

# Set a larger figure size
plt.figure(figsize=(10, 6))

# Use Seaborn's lineplot for better handling of lines and markers
sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')

# Use interpolated confidence interval with reduced alpha
plt.fill_between(fitted_and_forecasted.index, ci_interpolated_lower, ci_interpolated_upper, color='red', alpha=0.2, label='85% Confidence Interval')

# Set y-axis lower limit to 0
plt.ylim(bottom=0)

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data with Randomness')
plt.legend()
plt.show()

# Compute and print the covariance matrix, standard deviation, and MAE
summary = result.summary()
cov_matrix = pd.DataFrame(result.cov_params(), columns=result.params.index)
std_dev = pd.DataFrame(result.bse, columns=['Standard Deviation'])
mae = np.mean(np.abs(data['COUNT'] - result.fittedvalues))

# Convert covariance matrix to a list of lists
cov_matrix_list = cov_matrix.applymap(lambda x: f"{x:.4f}").values.tolist()
std_dev_list = std_dev.applymap(lambda x: f"{x:.4f}").values.tolist()

# Create a beautiful table
table_data = {
    'Covariance Matrix': cov_matrix_list,
    'Standard Deviation': std_dev_list,
    'Mean Absolute Error (MAE)': [f"{mae:.4f}"]
}

table = tabulate(table_data, headers='keys', tablefmt='pretty')
print(table)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from tabulate import tabulate

class ARIMAModel:
    def __init__(self, data, p=1, d=1, q=1, alpha=0.85, future_steps=12, noise_scale=1.5):
        self.data = data
        self.p = p
        self.d = d
        self.q = q
        self.alpha = alpha
        self.future_steps = future_steps
        self.noise_scale = noise_scale

    def fit_model(self):
        order = (self.p, self.d, self.q)
        model = ARIMA(self.data['COUNT'], order=order)
        result = model.fit()
        return result

    def generate_forecast_with_noise(self, result):
        forecast = result.get_forecast(steps=self.future_steps)
        randomness = np.random.normal(scale=self.noise_scale, size=self.future_steps)
        forecast_values_with_noise = forecast.predicted_mean + randomness
        return forecast, forecast_values_with_noise

    def interpolate_confidence_interval(self, result, common_index):
        forecast = result.get_forecast(steps=self.future_steps)
        ci = forecast.conf_int(alpha=self.alpha)
        common_index_truncated = common_index[:len(forecast.predicted_mean)]
        ci_interpolated_lower = np.interp(forecast.predicted_mean.index, common_index_truncated, ci.iloc[:, 0])
        ci_interpolated_upper = np.interp(forecast.predicted_mean.index, common_index_truncated, ci.iloc[:, 1])
        return ci_interpolated_lower, ci_interpolated_upper

def plot_results(data, fitted_and_forecasted, ci_interpolated_lower, ci_interpolated_upper):
    plt.figure(figsize=(10, 6))
    sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
    sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')
    plt.fill_between(fitted_and_forecasted.index, ci_interpolated_lower, ci_interpolated_upper, color='red', alpha=0.2, label='85% Confidence Interval')
    plt.ylim(bottom=0)
    plt.xlabel('Time Scale')
    plt.ylabel('Count')
    plt.title('ARIMA Model for Time Series Data with Randomness')
    plt.legend()
    plt.show()

def compute_statistics(result, data):
    summary = result.summary()
    cov_matrix = pd.DataFrame(result.cov_params(), columns=result.params.index)
    std_dev = pd.DataFrame(result.bse, columns=['Standard Deviation'])
    mae = np.mean(np.abs(data['COUNT'] - result.fittedvalues))
    
    cov_matrix_list = cov_matrix.applymap(lambda x: f"{x:.4f}").values.tolist()
    std_dev_list = std_dev.applymap(lambda x: f"{x:.4f}").values.tolist()

    table_data = {
        'Covariance Matrix': cov_matrix_list,
        'Standard Deviation': std_dev_list,
        'Mean Absolute Error (MAE)': [f"{mae:.4f}"]
    }

    table = tabulate(table_data, headers='keys', tablefmt='pretty')
    print(table)

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame
common_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))

arima_model = ARIMAModel(data)
result = arima_model.fit_model()
forecast, forecast_values_with_noise = arima_model.generate_forecast_with_noise(result)
ci_interpolated_lower, ci_interpolated_upper = arima_model.interpolate_confidence_interval(result, common_index)

plot_results(data, forecast.predicted_mean, ci_interpolated_lower, ci_interpolated_upper)
compute_statistics(result, data)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from tabulate import tabulate

class ARIMAModel:
    def __init__(self, data, p=1, d=1, q=1, alpha=0.85, future_steps=12, noise_scale=1.5):
        self.data = data
        self.p = p
        self.d = d
        self.q = q
        self.alpha = alpha
        self.future_steps = future_steps
        self.noise_scale = noise_scale

    def fit_model(self):
        order = (self.p, self.d, self.q)
        model = ARIMA(self.data['COUNT'], order=order)
        result = model.fit()
        return result

    def generate_forecast_with_noise(self, result):
        forecast = result.get_forecast(steps=self.future_steps)
        randomness = np.random.normal(scale=self.noise_scale, size=self.future_steps)
        forecast_values_with_noise = forecast.predicted_mean + randomness
        return forecast, forecast_values_with_noise

    def interpolate_confidence_interval(self, result, common_index):
        forecast = result.get_forecast(steps=self.future_steps)
        ci = forecast.conf_int(alpha=self.alpha)
        common_index_truncated = common_index[:len(forecast.predicted_mean)]
        ci_interpolated_lower = np.interp(forecast.predicted_mean.index, common_index_truncated, ci.iloc[:, 0])
        ci_interpolated_upper = np.interp(forecast.predicted_mean.index, common_index_truncated, ci.iloc[:, 1])
        return ci_interpolated_lower, ci_interpolated_upper

def plot_results(data, fitted_and_forecasted, ci_interpolated_lower, ci_interpolated_upper):
    plt.figure(figsize=(10, 6))
    sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
    
    # Adjust the x values for the fitted and forecasted data
    x_values = data['TIME_SCALE'].append(pd.Index(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))
    
    sns.lineplot(x=x_values, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')
    plt.fill_between(x_values, ci_interpolated_lower, ci_interpolated_upper, color='red', alpha=0.2, label='85% Confidence Interval')
    plt.ylim(bottom=0)
    plt.xlabel('Time Scale')
    plt.ylabel('Count')
    plt.title('ARIMA Model for Time Series Data with Randomness')
    plt.legend()
    plt.show()

def compute_statistics(result, data):
    summary = result.summary()
    cov_matrix = pd.DataFrame(result.cov_params(), columns=result.params.index)
    std_dev = pd.DataFrame(result.bse, columns=['Standard Deviation'])
    mae = np.mean(np.abs(data['COUNT'] - result.fittedvalues))
    
    cov_matrix_list = cov_matrix.applymap(lambda x: f"{x:.4f}").values.tolist()
    std_dev_list = std_dev.applymap(lambda x: f"{x:.4f}").values.tolist()

    table_data = {
        'Covariance Matrix': cov_matrix_list,
        'Standard Deviation': std_dev_list,
        'Mean Absolute Error (MAE)': [f"{mae:.4f}"]
    }

    table = tabulate(table_data, headers='keys', tablefmt='pretty')
    print(table)

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame
common_index = pd.Index(data['TIME_SCALE'].append(pd.Index(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps))))

arima_model = ARIMAModel(data)
result = arima_model.fit_model()
forecast, forecast_values_with_noise = arima_model.generate_forecast_with_noise(result)
ci_interpolated_lower, ci_interpolated_upper = arima_model.interpolate_confidence_interval(result, common_index)

plot_results(data, forecast.predicted_mean, ci_interpolated_lower, ci_interpolated_upper)
compute_statistics(result, data)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from tabulate import tabulate

class ARIMAModel:
    def __init__(self, data, p=1, d=1, q=1, alpha=0.85, future_steps=12, noise_scale=1.5):
        self.data = data
        self.p = p
        self.d = d
        self.q = q
        self.alpha = alpha
        self.future_steps = future_steps
        self.noise_scale = noise_scale

    def fit_model(self):
        order = (self.p, self.d, self.q)
        model = ARIMA(self.data['COUNT'], order=order)
        result = model.fit()
        return result

    def generate_forecast_with_noise(self, result):
        forecast = result.get_forecast(steps=self.future_steps)
        randomness = np.random.normal(scale=self.noise_scale, size=self.future_steps)
        forecast_values_with_noise = forecast.predicted_mean + randomness
        return forecast, forecast_values_with_noise

    def interpolate_confidence_interval(self, result, common_index):
        forecast = result.get_forecast(steps=self.future_steps)
        ci = forecast.conf_int(alpha=self.alpha)
        common_index_truncated = common_index[:len(forecast.predicted_mean)]
        ci_interpolated_lower = np.interp(forecast.predicted_mean.index, common_index_truncated, ci.iloc[:, 0])
        ci_interpolated_upper = np.interp(forecast.predicted_mean.index, common_index_truncated, ci.iloc[:, 1])
        return ci_interpolated_lower, ci_interpolated_upper

def plot_results(data, fitted_and_forecasted, ci_interpolated_lower, ci_interpolated_upper):
    plt.figure(figsize=(10, 6))
    sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
    
    # Adjust the x values for the fitted and forecasted data
    x_values = pd.Index(data['TIME_SCALE']).append(pd.Index(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))
    
    sns.lineplot(x=x_values, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')
    plt.fill_between(x_values, ci_interpolated_lower, ci_interpolated_upper, color='red', alpha=0.2, label='85% Confidence Interval')
    plt.ylim(bottom=0)
    plt.xlabel('Time Scale')
    plt.ylabel('Count')
    plt.title('ARIMA Model for Time Series Data with Randomness')
    plt.legend()
    plt.show()

def compute_statistics(result, data):
    summary = result.summary()
    cov_matrix = pd.DataFrame(result.cov_params(), columns=result.params.index)
    std_dev = pd.DataFrame(result.bse, columns=['Standard Deviation'])
    mae = np.mean(np.abs(data['COUNT'] - result.fittedvalues))
    
    cov_matrix_list = cov_matrix.applymap(lambda x: f"{x:.4f}").values.tolist()
    std_dev_list = std_dev.applymap(lambda x: f"{x:.4f}").values.tolist()

    table_data = {
        'Covariance Matrix': cov_matrix_list,
        'Standard Deviation': std_dev_list,
        'Mean Absolute Error (MAE)': [f"{mae:.4f}"]
    }

    table = tabulate(table_data, headers='keys', tablefmt='pretty')
    print(table)

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame
common_index = pd.Index(data['TIME_SCALE']).append(pd.Index(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))

arima_model = ARIMAModel(data)
result = arima_model.fit_model()
forecast, forecast_values_with_noise = arima_model.generate_forecast_with_noise(result)
ci_interpolated_lower, ci_interpolated_upper = arima_model.interpolate_confidence_interval(result, common_index)

plot_results(data, forecast.predicted_mean, ci_interpolated_lower, ci_interpolated_upper)
compute_statistics(result, data)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from tabulate import tabulate  # Import tabulate from the tabulate library

# Set Seaborn style
sns.set(style="whitegrid")

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 12
forecast = result.get_forecast(steps=future_steps)

# Extract upper and lower bounds of the confidence interval
ci = forecast.conf_int(alpha=0.85)  # Set alpha to 0.85 for 85% confidence interval

# Introduce randomness by adding white noise
np.random.seed(42)  # Set seed for reproducibility
randomness = np.random.normal(scale=1.5, size=future_steps)
forecast_values_with_noise = forecast.predicted_mean + randomness

# Use the common index between forecast values and confidence interval
common_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))

# Truncate common_index to match the length of forecast_values_with_noise
common_index_truncated = common_index[:len(forecast_values_with_noise)]

# Interpolate confidence interval to match the length of fitted and forecasted values
ci_interpolated_lower = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 0])
ci_interpolated_upper = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 1])

# Set a larger figure size
plt.figure(figsize=(10, 6))

# Use Seaborn's lineplot for better handling of lines and markers
sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')

# Use interpolated confidence interval with reduced alpha
plt.fill_between(fitted_and_forecasted.index, ci_interpolated_lower, ci_interpolated_upper, color='red', alpha=0.2, label='85% Confidence Interval')

# Set y-axis lower limit to 0
plt.ylim(bottom=0)

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data with Randomness')
plt.legend()
plt.show()

# Compute and print the covariance matrix, standard deviation, and MAE
summary = result.summary()
cov_matrix = pd.DataFrame(result.cov_params(), columns=result.params.index)
std_dev = pd.DataFrame(result.bse, columns=['Standard Deviation'])
mae = np.mean(np.abs(data['COUNT'] - result.fittedvalues))

# Convert covariance matrix to a list of lists
cov_matrix_list = cov_matrix.applymap(lambda x: f"{x:.4f}").values.tolist()
std_dev_list = std_dev.applymap(lambda x: f"{x:.4f}").values.tolist()

# Create a beautiful table
table_data = {
    'Covariance Matrix': cov_matrix_list,
    'Standard Deviation': std_dev_list,
    'Mean Absolute Error (MAE)': [f"{mae:.4f}"]
}

table = tabulate(table_data, headers='keys', tablefmt='pretty')
print(table)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from tabulate import tabulate

# Set Seaborn style
def set_seaborn_style():
    sns.set(style="whitegrid")

# Fit ARIMA model
def fit_arima_model(data, p=1, d=1, q=1):
    order = (p, d, q)
    model = ARIMA(data['COUNT'], order=order)
    result = model.fit()
    return result

# Generate ARIMA forecast
def generate_arima_forecast(result, future_steps=12, alpha=0.85):
    forecast = result.get_forecast(steps=future_steps)
    
    # Extract upper and lower bounds of the confidence interval
    ci = forecast.conf_int(alpha=alpha)
    
    # Introduce randomness by adding white noise
    np.random.seed(42)
    randomness = np.random.normal(scale=1.5, size=future_steps)
    forecast_values_with_noise = forecast.predicted_mean + randomness
    
    return forecast, ci, forecast_values_with_noise

# Prepare common index
def prepare_common_index(data, future_steps):
    common_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))
    return common_index

# Interpolate confidence interval
def interpolate_confidence_interval(ci, fitted_and_forecasted, common_index_truncated):
    ci_interpolated_lower = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 0])
    ci_interpolated_upper = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 1])
    return ci_interpolated_lower, ci_interpolated_upper

# Plot ARIMA results
def plot_arima_results(data, fitted_and_forecasted, ci_interpolated_lower, ci_interpolated_upper):
    plt.figure(figsize=(10, 6))
    sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
    sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')
    plt.fill_between(fitted_and_forecasted.index, ci_interpolated_lower, ci_interpolated_upper, color='red', alpha=0.2, label='85% Confidence Interval')
    plt.ylim(bottom=0)
    plt.xlabel('Time Scale')
    plt.ylabel('Count')
    plt.title('ARIMA Model for Time Series Data with Randomness')
    plt.legend()
    plt.show()

# Summarize ARIMA results
def summarize_arima_results(result, data):
    summary = result.summary()
    cov_matrix = pd.DataFrame(result.cov_params(), columns=result.params.index)
    std_dev = pd.DataFrame(result.bse, columns=['Standard Deviation'])
    mae = np.mean(np.abs(data['COUNT'] - result.fittedvalues))
    
    # Convert covariance matrix to a list of lists
    cov_matrix_list = cov_matrix.applymap(lambda x: f"{x:.4f}").values.tolist()
    std_dev_list = std_dev.applymap(lambda x: f"{x:.4f}").values.tolist()
    
    # Create a beautiful table
    table_data = {
        'Covariance Matrix': cov_matrix_list,
        'Standard Deviation': std_dev_list,
        'Mean Absolute Error (MAE)': [f"{mae:.4f}"]
    }

    table = tabulate(table_data, headers='keys', tablefmt='pretty')
    print(table)

# Main execution
if __name__ == "__main__":
    # Load data (replace this with your data loading logic)
    
    set_seaborn_style()
    
    result = fit_arima_model(data)
    
    future_steps = 12
    forecast, ci, forecast_values_with_noise = generate_arima_forecast(result, future_steps=future_steps)
    
    common_index = prepare_common_index(data, future_steps)
    
    common_index_truncated = common_index[:len(forecast_values_with_noise)]
    
    ci_interpolated_lower, ci_interpolated_upper = interpolate_confidence_interval(ci, result.fittedvalues, common_index_truncated)
    
    plot_arima_results(data, result.fittedvalues, ci_interpolated_lower, ci_interpolated_upper)
    
    summarize_arima_results(result, data)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from tabulate import tabulate  # Import tabulate from the tabulate library

# Set Seaborn style
sns.set(style="whitegrid")

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 12
forecast = result.get_forecast(steps=future_steps)

# Extract upper and lower bounds of the confidence interval
ci = forecast.conf_int(alpha=0.85)  # Set alpha to 0.85 for 85% confidence interval

# Introduce randomness by adding white noise
np.random.seed(42)  # Set seed for reproducibility
randomness = np.random.normal(scale=1.5, size=future_steps)
forecast_values_with_noise = forecast.predicted_mean + randomness

# Use the common index between forecast values and confidence interval
common_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))

# Truncate common_index to match the length of forecast_values_with_noise
common_index_truncated = common_index[:len(forecast_values_with_noise)]

# Interpolate confidence interval to match the length of fitted and forecasted values
ci_interpolated_lower = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 0])
ci_interpolated_upper = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 1])

# Set a larger figure size
plt.figure(figsize=(10, 6))

# Use Seaborn's lineplot for better handling of lines and markers
sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')

# Use interpolated confidence interval with reduced alpha
plt.fill_between(fitted_and_forecasted.index, ci_interpolated_lower, ci_interpolated_upper, color='red', alpha=0.2, label='85% Confidence Interval')

# Set y-axis lower limit to 0
plt.ylim(bottom=0)

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data with Randomness')
plt.legend()
plt.show()

# Compute and print the covariance matrix, standard deviation, and MAE
summary = result.summary()
cov_matrix = pd.DataFrame(result.cov_params(), columns=result.params.index)
std_dev = pd.DataFrame(result.bse, columns=['Standard Deviation'])
mae = np.mean(np.abs(data['COUNT'] - result.fittedvalues))

# Convert covariance matrix to a list of lists
cov_matrix_list = cov_matrix.applymap(lambda x: f"{x:.4f}").values.tolist()
std_dev_list = std_dev.applymap(lambda x: f"{x:.4f}").values.tolist()

# Create a beautiful table
table_data = {
    'Covariance Matrix': cov_matrix_list,
    'Standard Deviation': std_dev_list,
    'Mean Absolute Error (MAE)': [f"{mae:.4f}"]
}

table = tabulate(table_data, headers='keys', tablefmt='pretty')
print(table)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from tabulate import tabulate

def set_seaborn_style():
    sns.set(style="whitegrid")

def fit_arima_model(data, p=1, d=1, q=1):
    order = (p, d, q)
    model = ARIMA(data['COUNT'], order=order)
    result = model.fit()
    return result

def generate_arima_forecast(result, future_steps=12, alpha=0.85):
    forecast = result.get_forecast(steps=future_steps)
    ci = forecast.conf_int(alpha=alpha)
    np.random.seed(42)
    randomness = np.random.normal(scale=1.5, size=future_steps)
    forecast_values_with_noise = forecast.predicted_mean + randomness
    return forecast, ci, forecast_values_with_noise

def prepare_common_index(data, future_steps):
    common_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))
    return common_index

def interpolate_confidence_interval(ci, fitted_and_forecasted, common_index_truncated):
    ci_interpolated_lower = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 0])
    ci_interpolated_upper = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 1])
    return ci_interpolated_lower, ci_interpolated_upper

def plot_arima_results(data, fitted_and_forecasted, ci_interpolated_lower, ci_interpolated_upper):
    plt.figure(figsize=(10, 6))
    sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
    sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')
    plt.fill_between(fitted_and_forecasted.index, ci_interpolated_lower, ci_interpolated_upper, color='red', alpha=0.2, label='85% Confidence Interval')
    plt.ylim(bottom=0)
    plt.xlabel('Time Scale')
    plt.ylabel('Count')
    plt.title('ARIMA Model for Time Series Data with Randomness')
    plt.legend()
    plt.show()

def summarize_arima_results(result, data):
    summary = result.summary()
    cov_matrix = pd.DataFrame(result.cov_params(), columns=result.params.index)
    std_dev = pd.DataFrame(result.bse, columns=['Standard Deviation'])
    mae = np.mean(np.abs(data['COUNT'] - result.fittedvalues))
    
    cov_matrix_list = cov_matrix.applymap(lambda x: f"{x:.4f}").values.tolist()
    std_dev_list = std_dev.applymap(lambda x: f"{x:.4f}").values.tolist()
    
    table_data = {
        'Covariance Matrix': cov_matrix_list,
        'Standard Deviation': std_dev_list,
        'Mean Absolute Error (MAE)': [f"{mae:.4f}"]
    }

    table = tabulate(table_data, headers='keys', tablefmt='pretty')
    print(table)

# Main execution
if __name__ == "__main__":
    # Load data (replace this with your data loading logic)
    
    set_seaborn_style()
    
    result = fit_arima_model(data)
    
    future_steps = 12
    forecast, ci, forecast_values_with_noise = generate_arima_forecast(result, future_steps=future_steps)
    
    common_index = prepare_common_index(data, future_steps)
    
    common_index_truncated = common_index[:len(forecast_values_with_noise)]
    
    ci_interpolated_lower, ci_interpolated_upper = interpolate_confidence_interval(ci, result.fittedvalues, common_index_truncated)
    
    plot_arima_results(data, result.fittedvalues, ci_interpolated_lower, ci_interpolated_upper)
    
    summarize_arima_results(result, data)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from tabulate import tabulate

def set_seaborn_style():
    sns.set(style="whitegrid")

def fit_arima_model(data, p=1, d=1, q=1):
    order = (p, d, q)
    model = ARIMA(data['COUNT'], order=order)
    result = model.fit()
    return result

def generate_arima_forecast(result, future_steps=12, alpha=0.85):
    forecast = result.get_forecast(steps=future_steps)
    ci = forecast.conf_int(alpha=alpha)
    np.random.seed(42)
    randomness = np.random.normal(scale=1.5, size=future_steps)
    forecast_values_with_noise = forecast.predicted_mean + randomness
    return forecast, ci, forecast_values_with_noise

def prepare_common_index(data, future_steps):
    # Start the common index from the maximum value of the original time scale
    common_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))
    return common_index

def interpolate_confidence_interval(ci, fitted_and_forecasted, common_index_truncated):
    ci_interpolated_lower = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 0])
    ci_interpolated_upper = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 1])
    return ci_interpolated_lower, ci_interpolated_upper

def plot_arima_results(data, fitted_and_forecasted, ci_interpolated_lower, ci_interpolated_upper):
    plt.figure(figsize=(10, 6))
    sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
    sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')
    plt.fill_between(fitted_and_forecasted.index, ci_interpolated_lower, ci_interpolated_upper, color='red', alpha=0.2, label='85% Confidence Interval')
    plt.ylim(bottom=0)
    plt.xlabel('Time Scale')
    plt.ylabel('Count')
    plt.title('ARIMA Model for Time Series Data with Randomness')
    plt.legend()
    plt.show()

def summarize_arima_results(result, data):
    summary = result.summary()
    cov_matrix = pd.DataFrame(result.cov_params(), columns=result.params.index)
    std_dev = pd.DataFrame(result.bse, columns=['Standard Deviation'])
    mae = np.mean(np.abs(data['COUNT'] - result.fittedvalues))
    
    cov_matrix_list = cov_matrix.applymap(lambda x: f"{x:.4f}").values.tolist()
    std_dev_list = std_dev.applymap(lambda x: f"{x:.4f}").values.tolist()
    
    table_data = {
        'Covariance Matrix': cov_matrix_list,
        'Standard Deviation': std_dev_list,
        'Mean Absolute Error (MAE)': [f"{mae:.4f}"]
    }

    table = tabulate(table_data, headers='keys', tablefmt='pretty')
    print(table)

# Main execution
if __name__ == "__main__":
    # Load data (replace this with your data loading logic)
    
    set_seaborn_style()
    
    result = fit_arima_model(data)
    
    future_steps = 12
    forecast, ci, forecast_values_with_noise = generate_arima_forecast(result, future_steps=future_steps)
    
    common_index = prepare_common_index(data, future_steps)
    
    common_index_truncated = common_index[:len(forecast_values_with_noise)]
    
    ci_interpolated_lower, ci_interpolated_upper = interpolate_confidence_interval(ci, result.fittedvalues, common_index_truncated)
    
    plot_arima_results(data, result.fittedvalues, ci_interpolated_lower, ci_interpolated_upper)
    
    summarize_arima_results(result, data)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from tabulate import tabulate

def set_seaborn_style():
    sns.set(style="whitegrid")

def fit_arima_model(data, p=1, d=1, q=1):
    order = (p, d, q)
    model = ARIMA(data['COUNT'], order=order)
    result = model.fit()
    return result

def generate_arima_forecast(result, future_steps=12, alpha=0.85):
    forecast = result.get_forecast(steps=future_steps)
    ci = forecast.conf_int(alpha=alpha)
    np.random.seed(42)
    randomness = np.random.normal(scale=1.5, size=future_steps)
    forecast_values_with_noise = forecast.predicted_mean + randomness
    return forecast, ci, forecast_values_with_noise

def prepare_common_index(data, future_steps):
    # Start the common index from the beginning of the original time scale
    common_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))
    return common_index

def interpolate_confidence_interval(ci, fitted_and_forecasted, common_index_truncated):
    ci_interpolated_lower = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 0])
    ci_interpolated_upper = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 1])
    return ci_interpolated_lower, ci_interpolated_upper

def plot_arima_results(data, fitted_and_forecasted, ci_interpolated_lower, ci_interpolated_upper):
    plt.figure(figsize=(10, 6))
    sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
    sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')
    plt.fill_between(fitted_and_forecasted.index, ci_interpolated_lower, ci_interpolated_upper, color='red', alpha=0.2, label='85% Confidence Interval')
    plt.ylim(bottom=0)
    plt.xlabel('Time Scale')
    plt.ylabel('Count')
    plt.title('ARIMA Model for Time Series Data with Randomness')
    plt.legend()
    plt.show()

def summarize_arima_results(result, data):
    summary = result.summary()
    cov_matrix = pd.DataFrame(result.cov_params(), columns=result.params.index)
    std_dev = pd.DataFrame(result.bse, columns=['Standard Deviation'])
    mae = np.mean(np.abs(data['COUNT'] - result.fittedvalues))
    
    cov_matrix_list = cov_matrix.applymap(lambda x: f"{x:.4f}").values.tolist()
    std_dev_list = std_dev.applymap(lambda x: f"{x:.4f}").values.tolist()
    
    table_data = {
        'Covariance Matrix': cov_matrix_list,
        'Standard Deviation': std_dev_list,
        'Mean Absolute Error (MAE)': [f"{mae:.4f}"]
    }

    table = tabulate(table_data, headers='keys', tablefmt='pretty')
    print(table)

# Main execution
if __name__ == "__main__":
    # Load data (replace this with your data loading logic)
    
    set_seaborn_style()
    
    result = fit_arima_model(data)
    
    future_steps = 12
    forecast, ci, forecast_values_with_noise = generate_arima_forecast(result, future_steps=future_steps)
    
    common_index = prepare_common_index(data, future_steps)
    
    common_index_truncated = common_index[:len(data) + future_steps]  # Adjust to extend the index until the end of original data
    
    ci_interpolated_lower, ci_interpolated_upper = interpolate_confidence_interval(ci, result.fittedvalues, common_index_truncated)
    
    plot_arima_results(data, result.fittedvalues, ci_interpolated_lower, ci_interpolated_upper)
    
    summarize_arima_results(result, data)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from tabulate import tabulate  # Import tabulate from the tabulate library

# Set Seaborn style
sns.set(style="whitegrid")

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 1  # Example values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Number of future steps to extrapolate
future_steps = 12
forecast = result.get_forecast(steps=future_steps)

# Extract upper and lower bounds of the confidence interval
ci = forecast.conf_int(alpha=0.85)  # Set alpha to 0.85 for 85% confidence interval

# Introduce randomness by adding white noise
np.random.seed(42)  # Set seed for reproducibility
randomness = np.random.normal(scale=1.5, size=future_steps)
forecast_values_with_noise = forecast.predicted_mean + randomness

# Use the common index between forecast values and confidence interval
common_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))

# Truncate common_index to match the length of forecast_values_with_noise
common_index_truncated = common_index[:len(forecast_values_with_noise)]

# Interpolate confidence interval to match the length of fitted and forecasted values
ci_interpolated_lower = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 0])
ci_interpolated_upper = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 1])

# Set a larger figure size
plt.figure(figsize=(10, 6))

# Use Seaborn's lineplot for better handling of lines and markers
sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')

# Use interpolated confidence interval with reduced alpha
plt.fill_between(fitted_and_forecasted.index, ci_interpolated_lower, ci_interpolated_upper, color='red', alpha=0.2, label='85% Confidence Interval')

# Set y-axis lower limit to 0
plt.ylim(bottom=0)

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data with Randomness')
plt.legend()
plt.show()

# Compute and print the covariance matrix, standard deviation, and MAE
summary = result.summary()
cov_matrix = pd.DataFrame(result.cov_params(), columns=result.params.index)
std_dev = pd.DataFrame(result.bse, columns=['Standard Deviation'])
mae = np.mean(np.abs(data['COUNT'] - result.fittedvalues))

# Convert covariance matrix to a list of lists
cov_matrix_list = cov_matrix.applymap(lambda x: f"{x:.4f}").values.tolist()
std_dev_list = std_dev.applymap(lambda x: f"{x:.4f}").values.tolist()

# Create a beautiful table
table_data = {
    'Covariance Matrix': cov_matrix_list,
    'Standard Deviation': std_dev_list,
    'Mean Absolute Error (MAE)': [f"{mae:.4f}"]
}

table = tabulate(table_data, headers='keys', tablefmt='pretty')
print(table)


In [ ]:
import pandas as pd 

# Load time series data from CSV file
file_path = "/Users/igormol/Desktop/John/KITPL1A.csv"
data = pd.read_csv(file_path)

In [ ]:
data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from tabulate import tabulate

# Set Seaborn style
sns.set(style="whitegrid")

# Assuming your data frame is named 'data'
# If not, replace 'data' with the actual name of your data frame

# Fit ARIMA model
p, d, q = 1, 1, 2  # Adjusted values for p, d, q
order = (p, d, q)
model = ARIMA(data['COUNT'], order=order)
result = model.fit()

# Check residuals
residuals = result.resid
plt.plot(residuals)
plt.title('Residuals Plot')
plt.show()

# Continue with the rest of your code...

# Number of future steps to extrapolate
future_steps = 12
forecast = result.get_forecast(steps=future_steps)

# Extract upper and lower bounds of the confidence interval
ci = forecast.conf_int(alpha=0.85)  # Set alpha to 0.85 for 85% confidence interval

# Introduce randomness by adding white noise
np.random.seed(42)  # Set seed for reproducibility
randomness = np.random.normal(scale=1.5, size=future_steps)
forecast_values_with_noise = forecast.predicted_mean + randomness

# Use the common index between forecast values and confidence interval
common_index = pd.Index(list(data['TIME_SCALE']) + list(range(data['TIME_SCALE'].max() + 1, data['TIME_SCALE'].max() + 1 + future_steps)))

# Truncate common_index to match the length of forecast_values_with_noise
common_index_truncated = common_index[:len(forecast_values_with_noise)]

# Interpolate confidence interval to match the length of fitted and forecasted values
ci_interpolated_lower = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 0])
ci_interpolated_upper = np.interp(fitted_and_forecasted.index, common_index_truncated, ci.iloc[:, 1])

# Set a larger figure size
plt.figure(figsize=(10, 6))

# Use Seaborn's lineplot for better handling of lines and markers
sns.lineplot(x=data['TIME_SCALE'], y=data['COUNT'], label='Original Data')
sns.lineplot(x=fitted_and_forecasted.index, y=fitted_and_forecasted, color='red', label='Fitted Model and Future Extrapolation with Randomness')

# Use interpolated confidence interval with reduced alpha
plt.fill_between(fitted_and_forecasted.index, ci_interpolated_lower, ci_interpolated_upper, color='red', alpha=0.2, label='85% Confidence Interval')

# Set y-axis lower limit to 0
plt.ylim(bottom=0)

plt.xlabel('Time Scale')
plt.ylabel('Count')
plt.title('ARIMA Model for Time Series Data with Randomness')
plt.legend()
plt.show()

# Compute and print the covariance matrix, standard deviation, and MAE
summary = result.summary()
cov_matrix = pd.DataFrame(result.cov_params(), columns=result.params.index)
std_dev = pd.DataFrame(result.bse, columns=['Standard Deviation'])
mae = np.mean(np.abs(data['COUNT'] - result.fittedvalues))

# Convert covariance matrix to a list of lists
cov_matrix_list = cov_matrix.applymap(lambda x: f"{x:.4f}").values.tolist()
std_dev_list = std_dev.applymap(lambda x: f"{x:.4f}").values.tolist()

# Create a beautiful table
table_data = {
    'Covariance Matrix': cov_matrix_list,
    'Standard Deviation': std_dev_list,
    'Mean Absolute Error (MAE)': [f"{mae:.4f}"]
}

table = tabulate(table_data, headers='keys', tablefmt='pretty')
print(table)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
import seaborn as sns
import matplotlib.pyplot as plt
from tabulate import tabulate

class TimeSeriesModel:
    def __init__(self, data, window_size=1, months_to_predict=12, randomness_factor=0):
        """
        Initialize the TimeSeriesModel with input data and configuration parameters.

        Parameters:
        - data: Input time-series data as a Pandas DataFrame.
        - window_size: Size of the sliding window for predictions.
        - months_to_predict: Number of months to extrapolate into the future.
        - randomness_factor: Factor to introduce randomness in future extrapolation.
        """
        self.data = data
        self.window_size = window_size
        self.months_to_predict = months_to_predict
        self.randomness_factor = randomness_factor
        self.predictions = []  # List to store model predictions
        self.future_extrapolation = []  # List to store future extrapolation time points

    def preprocess_data(self):
        """
        Preprocess the input data by sorting, removing NaN values, and filling missing values.
        """
        # Sort the data based on the 'TIME_SCALE' column
        self.data = self.data.sort_values(by='TIME_SCALE')
        # Drop rows where 'COUNT' is NaN
        self.data = self.data.dropna(subset=['COUNT'])

        # Fill NaN values in 'COUNT' with the mean of the column
        if self.data['COUNT'].isnull().sum() > 0:
            self.data['COUNT'].fillna(self.data['COUNT'].mean(), inplace=True)

    def train_model(self, X, y):
        """
        Train a Support Vector Regression (SVR) model with an RBF kernel.

        Parameters:
        - X: Input features.
        - y: Target variable.

        Returns:
        Trained SVR model.
        """
        model = SVR(kernel='rbf', gamma='scale')
        model.fit(X, y)
        return model

    def generate_predictions(self, X, model):
        """
        Generate predictions using a trained model.

        Parameters:
        - X: Input features for prediction.
        - model: Trained model.

        Returns:
        Array of predictions.
        """
        return model.predict(X)

    def sliding_window_predictions(self):
        """
        Perform sliding window predictions on the input time series data.
        """
        # Iterate through the data using a sliding window approach
        for i in range(len(self.data) - self.window_size + 1):
            # Extract a window of data
            window_data = self.data.iloc[i:i + self.window_size]
            # Prepare features (X) and target variable (y) for the model
            X = window_data['TIME_SCALE'].values.reshape(-1, 1)
            y = window_data['COUNT'].values

            # Skip if there are NaN values in the target variable
            if np.isnan(y).any():
                continue

            # Train the model using the current window
            model = self.train_model(X, y)

            # If there are more data points, predict the next time point and store the prediction
            if i + self.window_size < len(self.data):
                next_time_point = self.data.iloc[i + self.window_size]['TIME_SCALE']
                self.future_extrapolation.append(next_time_point)
                prediction = self.generate_predictions([[next_time_point]], model)
                self.predictions.append(prediction[0])

                # Add 'PREDICTION' column to the DataFrame
                self.data.at[i + self.window_size, 'PREDICTION'] = prediction[0]

    def extrapolate_future(self):
        """
        Extrapolate future data points for a specified number of months.
        """
        # Iterate for the specified number of months
        for _ in range(self.months_to_predict):
            # Prepare features (X) and target variable (y) for the model using all available data
            X = self.data['TIME_SCALE'].values.reshape(-1, 1)
            y = self.data['COUNT'].values

            # Skip if there are NaN values in the target variable
            if np.isnan(y).any():
                continue

            # Train the model using all available data
            model = self.train_model(X, y)

            # Predict the next time point and add a random factor
            next_time_point = self.future_extrapolation[-1] + 1
            self.future_extrapolation.append(next_time_point)
            prediction = self.generate_predictions([[next_time_point]], model)
            self.predictions.append(prediction[0] + np.random.normal(scale=self.randomness_factor))

    def calculate_metrics(self):
        """
        Calculate Mean Absolute Error (MAE), covariance matrix, and standard deviation.
        """
        mae = mean_absolute_error(self.data['COUNT'][:len(self.data)], self.predictions[:len(self.data)])
        covariance_matrix = np.cov(self.data['COUNT'][:len(self.data)], self.predictions[:len(self.data)])
        std_deviation = np.std(self.data['COUNT'][:len(self.data)] - self.predictions[:len(self.data)])
        return mae, covariance_matrix, std_deviation

    def plot_results(self):
        """
        Plot the original data, model predictions, and future extrapolation.
        """
        sns.set(style="whitegrid")
        plt.figure(figsize=(12, 6))

        combined_time = np.concatenate((self.data['TIME_SCALE'], self.future_extrapolation))
        combined_predictions = np.concatenate((self.data['PREDICTION'], self.predictions))

        sns.lineplot(x=self.data['TIME_SCALE'], y=self.data['COUNT'], label='Original Data', color='blue')
        sns.lineplot(x=combined_time, y=combined_predictions,
                     label='Model Prediction and Future Extrapolation', linestyle='dashed', color='green', marker='o')

        plt.xlabel('Time Scale')
        plt.ylabel('Count')
        plt.title(f'Time Series Modeling with Sliding Window and {self.months_to_predict} Months Future Extrapolation (SVM)')
        plt.legend()
        plt.show()

    def display_results_table(self, mae, covariance_matrix, std_deviation):
        """
        Display a table of calculated metrics.

        Parameters:
        - mae: Mean Absolute Error.
        - covariance_matrix: Covariance matrix.
        - std_deviation: Standard deviation.
        """
        results_table = pd.DataFrame({
            'Metric': ['Mean Absolute Error (MAE)', 'Covariance Matrix', 'Standard Deviation'],
            'Value': [mae, covariance_matrix, std_deviation]
        })

        print(tabulate(results_table, headers='keys', tablefmt='fancy_grid'))


def main():
    # Assuming your DataFrame is named 'data'
    # If not, replace 'data' with your actual DataFrame name
    time_series_model = TimeSeriesModel(data)
    time_series_model.preprocess_data()
    time_series_model.sliding_window_predictions()
    time_series_model.extrapolate_future()
    mae, covariance_matrix, std_deviation = time_series_model.calculate_metrics()
    time_series_model.plot_results()
    time_series_model.display_results_table(mae, covariance_matrix, std_deviation)


if __name__ == "__main__":
    main()


In [ ]:
data.info()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Assuming 'data' is your original data frame
# You may need to install the seaborn library using: pip install seaborn
import seaborn as sns

# Generate synthetic data
np.random.seed(42)
data = pd.DataFrame({
    'TIME_SCALE': np.arange(1, 14),
    'COUNT': np.random.randint(10, 50, size=13)
})

# Function to detect and substitute outliers using Z-score
def monte_carlo_outlier_detection(data_frame, column, threshold=3):
    z_scores = np.abs((data_frame[column] - data_frame[column].mean()) / data_frame[column].std())
    outliers = z_scores > threshold
    data_frame.loc[outliers, column] = np.nan
    return data_frame

# Plot original data
sns.scatterplot(x='TIME_SCALE', y='COUNT', data=data, label='Original Data')
plt.title('Original Data')
plt.show()

# Apply Monte Carlo outlier detection
new_data = data.copy()
new_data = monte_carlo_outlier_detection(new_data, 'COUNT')

# Plot cleaned data
sns.scatterplot(x='TIME_SCALE', y='COUNT', data=new_data, label='Cleaned Data')
plt.title('Cleaned Data')
plt.show()

# Save the resulting time series in a data frame called "new_data"
new_data.reset_index(drop=True, inplace=True)

# Display the new_data
print("Original Data:")
print(data)
print("\nCleaned Data:")
print(new_data)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Assuming 'data' is your original data frame
# You may need to install the seaborn library using: pip install seaborn
import seaborn as sns

# Generate synthetic data
np.random.seed(42)
data = pd.DataFrame({
    'TIME_SCALE': np.arange(1, 14),
    'COUNT': np.random.randint(10, 50, size=13)
})

# Function to detect and substitute outliers using Z-score
def monte_carlo_outlier_detection(data_frame, column, threshold=1):
    z_scores = np.abs((data_frame[column] - data_frame[column].mean()) / data_frame[column].std())
    outliers = z_scores > threshold
    data_frame.loc[outliers, column] = np.nan
    return data_frame

# Plot original and cleaned data on the same plot
sns.scatterplot(x='TIME_SCALE', y='COUNT', data=data, label='Original Data')
new_data = data.copy()
new_data = monte_carlo_outlier_detection(new_data, 'COUNT')
sns.scatterplot(x='TIME_SCALE', y='COUNT', data=new_data, label='Cleaned Data')

plt.title('Original vs. Cleaned Data')
plt.legend()
plt.show()

# Save the resulting time series in a data frame called "new_data"
new_data.reset_index(drop=True, inplace=True)

# Display the new_data
print("Original Data:")
print(data)
print("\nCleaned Data:")
print(new_data)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Generate synthetic data
def generate_synthetic_data():
    np.random.seed(42)
    data = pd.DataFrame({
        'TIME_SCALE': np.arange(1, 14),
        'COUNT': np.random.randint(10, 50, size=13)
    })
    return data

# Function to detect and substitute outliers using Z-score
def monte_carlo_outlier_detection(data_frame, column, threshold=3):
    z_scores = np.abs((data_frame[column] - data_frame[column].mean()) / data_frame[column].std())
    outliers = z_scores > threshold
    data_frame.loc[outliers, column] = np.nan
    return data_frame

# Plot original and cleaned data on the same plot
def plot_original_vs_cleaned(original_data, cleaned_data):
    sns.scatterplot(x='TIME_SCALE', y='COUNT', data=original_data, label='Original Data')
    sns.scatterplot(x='TIME_SCALE', y='COUNT', data=cleaned_data, label='Cleaned Data')
    plt.title('Original vs. Cleaned Data')
    plt.legend()
    plt.show()

# Save the resulting time series in a data frame called "new_data"
def save_cleaned_data(original_data, column):
    new_data = original_data.copy()
    new_data = monte_carlo_outlier_detection(new_data, column)
    new_data.reset_index(drop=True, inplace=True)
    return new_data

# Display the data frames
def display_data_frames(original_data, cleaned_data):
    print("Original Data:")
    print(original_data)
    print("\nCleaned Data:")
    print(cleaned_data)

# Main function to execute the steps
def main():
    original_data = generate_synthetic_data()
    cleaned_data = save_cleaned_data(original_data, 'COUNT')
    plot_original_vs_cleaned(original_data, cleaned_data)
    display_data_frames(original_data, cleaned_data)

if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Generate synthetic data
def generate_synthetic_data():
    np.random.seed(42)
    data = pd.DataFrame({
        'TIME_SCALE': np.arange(1, 14),
        'COUNT': np.random.randint(10, 50, size=13)
    })
    return data

# Function to detect and substitute outliers using Z-score
def monte_carlo_outlier_detection(data_frame, column, threshold=3):
    z_scores = np.abs((data_frame[column] - data_frame[column].mean()) / data_frame[column].std())
    outliers = z_scores > threshold
    data_frame.loc[outliers, column] = np.nan
    return data_frame

# Plot original and cleaned data on the same plot
def plot_original_vs_cleaned(original_data, cleaned_data):
    sns.set(style="whitegrid")
    plt.figure(figsize=(10, 6))

    # Plot original data
    sns.scatterplot(x='TIME_SCALE', y='COUNT', data=original_data, label='Original Data', color='skyblue')

    # Plot cleaned data
    sns.scatterplot(x='TIME_SCALE', y='COUNT', data=cleaned_data, label='Cleaned Data', color='salmon', marker='X')

    plt.title('Original vs. Cleaned Data')
    plt.xlabel('Time Scale')
    plt.ylabel('Count')
    plt.legend()
    plt.show()

# Save the resulting time series in a data frame called "new_data"
def save_cleaned_data(original_data, column):
    new_data = original_data.copy()
    new_data = monte_carlo_outlier_detection(new_data, column)
    new_data.reset_index(drop=True, inplace=True)
    return new_data

# Display the data frames
def display_data_frames(original_data, cleaned_data):
    pd.set_option('display.float_format', '{:.2f}'.format)
    
    print("\nOriginal Data:")
    print(original_data.to_string(index=False))

    print("\nCleaned Data:")
    print(cleaned_data.to_string(index=False))

# Main function to execute the steps
def main():
    original_data = generate_synthetic_data()
    cleaned_data = save_cleaned_data(original_data, 'COUNT')
    plot_original_vs_cleaned(original_data, cleaned_data)
    display_data_frames(original_data, cleaned_data)

if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Generate synthetic data
def generate_synthetic_data():
    np.random.seed(42)
    data = pd.DataFrame({
        'TIME_SCALE': np.arange(1, 14),
        'COUNT': np.random.randint(10, 50, size=13)
    })
    return data

# Function to detect and substitute outliers using Z-score
def monte_carlo_outlier_detection(data_frame, column, threshold=1):
    z_scores = np.abs((data_frame[column] - data_frame[column].mean()) / data_frame[column].std())
    outliers = z_scores > threshold
    data_frame.loc[outliers, column] = np.nan
    return data_frame

# Plot original and cleaned data on the same plot
def plot_original_vs_cleaned(original_data, cleaned_data):
    sns.set(style="whitegrid")
    plt.figure(figsize=(10, 6))

    # Plot original data
    sns.scatterplot(x='TIME_SCALE', y='COUNT', data=original_data, label='Original Data', color='skyblue')

    # Plot cleaned data
    sns.scatterplot(x='TIME_SCALE', y='COUNT', data=cleaned_data, label='Cleaned Data', color='salmon', marker='X')

    plt.title('Original vs. Cleaned Data')
    plt.xlabel('Time Scale')
    plt.ylabel('Count')
    plt.legend()
    plt.show()

# Save the resulting time series in a data frame called "new_data"
def save_cleaned_data(original_data, column):
    new_data = original_data.copy()
    new_data = monte_carlo_outlier_detection(new_data, column)
    new_data.reset_index(drop=True, inplace=True)
    return new_data

# Display the data frames with improved formatting
def display_data_frames(original_data, cleaned_data):
    pd.set_option('display.precision', 2)
    
    print("\nOriginal Data:")
    print(original_data.to_string(index=False))

    print("\nCleaned Data:")
    print(cleaned_data.to_string(index=False))

# Main function to execute the steps
def main():
    original_data = generate_synthetic_data()
    cleaned_data = save_cleaned_data(original_data, 'COUNT')
    plot_original_vs_cleaned(original_data, cleaned_data)
    display_data_frames(original_data, cleaned_data)

if __name__ == "__main__":
    main()
